# **Jio CareOps Network Issue & Customer Support Analytics**

### Individual Project Notebook | Python · Pandas · Data Quality · Operations Analytics

## Project summary

**Project objective:** Build a reproducible Python and Pandas workflow that combines customer complaints, customer and cell-site reference data, network incidents, recharge events, and service requests into one complaint-level investigation report. The workflow identifies operational risk signals, assigns transparent P0–P3 priorities, recommends next actions, and exports reports for review.

**GitHub repository:** [Jio CareOps – Network Issue & Customer Support Analytics](https://github.com/Darpan2000-web/Jio-CareOps-Network-Issue-Customer-Support-Analytics)

**Author:** Darpan Banerjee

### Business problem
Customer Care and Network Operations need a consistent way to identify complaints that may require urgent network investigation, recharge/refund follow-up, service-request escalation, or repeat-customer attention. Looking at separate source files independently can hide relationships and can create incorrect counts when one-to-many tables are joined directly.

### Project approach
1. Inspect and profile the complaint baseline.
2. Load customer, site, network, recharge, and service-request sources.
3. Clean and standardize records while preserving data-quality flags.
4. Aggregate one-to-many operational records before joining them to complaints.
5. Build a complaint-level master dataset with one row per unique complaint.
6. Apply four explainable risk signals and a rule-based P0–P3 priority.
7. Export a detailed complaint report and an operations summary workbook.

### Dataset and scope note
This is a **synthetic training dataset** supplied for this learning project. It is not actual Jio customer, payment, or network-operation data. Results describe only the supplied sample and must not be interpreted as real-world Jio performance.

### Important analytical assumptions
- `complaint_id` is the complaint-level primary key.
- Network incidents are aggregated by `cell_site_id`; recharge events and service requests are aggregated by `complaint_id` before merging.
- The service-resolution review threshold of **more than 24 hours** is a project assumption used for demonstration, not a verified official Jio SLA.
- Priority is a transparent rule-based triage aid, not a predictive machine-learning model. Operational teams should validate the underlying case before taking action.
- Missing or unknown source values are retained and flagged where possible rather than silently treated as confirmed outcomes.

### Expected deliverables
- `jio_complaint_priority_report.csv` — complaint-level priority queue.
- `jio_careops_operations_summary.xlsx` — priority, circle, risk, action, and business-finding summaries.


## **Your Engineering Handover**

You have joined the Jio Customer Care and Network Operations team as a **Junior Python Developer**. The CareOps lead spends hours switching between customer complaint records, network incident reports, recharge failure logs, and service requests to understand which cases need attention. You have been asked to build a reproducible **customer complaint and network issue analysis workflow** in this notebook.

Your deliverable is **not an app or dashboard**. It is a trustworthy set of cleaned data tables, a complaint-level investigation report, and a concise evidence-based handover to Customer Care and Network Operations.

**How to work:** Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

**Input files (only three):** `jio_complaints_master.csv` · `jio_incident_histories.xlsx` · `jio_service_requests.json`  
**Data:** Fictional training records; not actual Jio customer or network operations records.

> **Important:** The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks.

### **Important Note: Guidelines for Using AI Tools**



You are allowed to use AI tools such as ChatGPT to understand concepts, debug errors, and get assistance with **specific code snippets or individual problems**.

However, **using AI to generate the complete project solution in one go is not permitted**.

You are expected to develop the solution milestone by milestone, apply your understanding of Python and Pandas, and independently verify your code and outputs.

**Remember:** The goal of this project is not just to complete the tasks but to develop practical problem-solving, debugging, and data analysis skills that are expected in real-world development roles.

## **Workspace setup · Data intake**
Run the setup cell and upload the **three input files** when requested. Files stay in your current Colab session; if the runtime restarts, upload them again. Keep original filenames unchanged.

[Link to Dataset](https://drive.google.com/file/d/1hgqvHzcqbEDG5e_sjSXjPljwV2_u7noh/view?usp=sharing)

In [1]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import json
import zipfile
DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['complaints.csv', 'network_issue_reports.xlsx', 'careops_reference.json']
ARCHIVE = 'jio_careops_project_datasets.zip'
if any(not (DATA_DIR/name).exists() for name in REQUIRED_FILES) and (DATA_DIR/ARCHIVE).is_file():
 with zipfile.ZipFile(DATA_DIR/ARCHIVE) as z:
  names=z.namelist()
  if sorted(names)!=sorted(REQUIRED_FILES):
   raise ValueError(f'Dataset ZIP must contain exactly the three expected files: {REQUIRED_FILES}; found {names}')
  z.extractall(DATA_DIR)
if any(not (DATA_DIR/name).exists() for name in REQUIRED_FILES):
 try:
  from google.colab import files
  print('Upload jio_careops_project_datasets.zip (containing CSV, XLSX and JSON).')
  files.upload()
 except ImportError:
  print('Place the dataset ZIP or three extracted files in',DATA_DIR)
 if (DATA_DIR/ARCHIVE).is_file():
  with zipfile.ZipFile(DATA_DIR/ARCHIVE) as z:
   if sorted(z.namelist())!=sorted(REQUIRED_FILES):
    raise ValueError('Unexpected files in dataset archive')
   z.extractall(DATA_DIR)
missing=[name for name in REQUIRED_FILES if not (DATA_DIR/name).exists()]
if missing: raise FileNotFoundError(f'Missing dataset files: {missing}')
print('Verified the three required Jio sources:', ', '.join(REQUIRED_FILES))

Upload jio_careops_project_datasets.zip (containing CSV, XLSX and JSON).


Saving jio_careops_project_datasets.zip to jio_careops_project_datasets.zip
Verified the three required Jio sources: complaints.csv, network_issue_reports.xlsx, careops_reference.json


## Milestone 01 · Establish the complaint baseline

**From the team:** The CareOps lead wants a transparent complaint intake audit before any triage recommendations are trusted.

**Your responsibility:** Load complaints.csv; inspect volume, unique complaint and customer IDs, missing values, and leading raw categories.

**Handover evidence:** Keep `complaints_df`; display shape, sample, duplicates and missing-value counts.

**Reasoning:** The complaint table is the central fact table, so its row count, key uniqueness, missing values, and raw categories must be understood before any transformation. This baseline makes later cleaning decisions measurable and helps detect accidental row loss.

In [2]:
# Milestone 01 — Establish the complaint baseline
# Task 1: Load complaints.csv; inspect volume, unique complaint and customer IDs, missing values, and leading raw categories.
# Task 2: Produce this required evidence: Keep `complaints_df`; display shape, sample, duplicates and missing-value counts.
# Implement your code below and display the requested outputs.

# MILESTONE 01: Complaint Data Assessment

# Task 1: Load dataset
complaints_df = pd.read_csv(DATA_DIR / 'complaints.csv')

# Task 2: Dataset dimensions
print("Dataset Shape:", complaints_df.shape)

# Task 3: First five records
display(complaints_df.head())

# Task 4: Dataset information
complaints_df.info()

# Task 5: Unique complaint and customer count
print("Unique Complaints:",
      complaints_df['complaint_id'].nunique())

print("Unique Customers:",
      complaints_df['customer_id'].nunique())

# Task 6: Duplicate complaint records
print("Duplicate Complaint IDs:",
      complaints_df['complaint_id'].duplicated().sum())

# Task 7: Missing value analysis
missing_summary = complaints_df.isnull().sum()

display(missing_summary[missing_summary > 0])

# Task 8: Complaint category analysis
display(complaints_df['issue_category'].value_counts(dropna=False))


Dataset Shape: (5000, 15)


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,complaint_status,sentiment_tag,network_type,device_type,amount_impacted
0,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,NaN,SIM activation or verification is pending afte...,Open,Urgent,VoLTE,JioPhone,799.0
1,CMP000002,CUST00793,SITE0461,2026-02-05,19:16,Ahmedabad,Gujarat,Retail Store,NaN,SIM activation or verification is pending afte...,NaN,Urgent,VoLTE,iOS,399.0
2,CMP000003,CUST00931,SITE0550,2026-02-09,20:46,Hyderabad,Andhra Pradesh & Telangana,Call Center,NaN,Fiber connection outage reported for apartment...,In Progress,Angry,5G,JioPhone,199.0
3,CMP000004,CUST00043,SITE0443,2026-02-18,13:45,Ahmedabad,Gujarat,Email,internet not working,Customer says calls drop during office hours.,Open,Frustrated,VoLTE,iOS,399.0
4,CMP000005,CUST00723,SITE0214,2026-02-01,23:28,Chennai,Tamil Nadu,Social Media,payment deducted recharge not done,Fiber connection outage reported for apartment...,closeD,Urgent,Fiber,JioPhone,399.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   complaint_id       5000 non-null   object 
 1   customer_id        5000 non-null   object 
 2   cell_site_id       5000 non-null   object 
 3   complaint_date     4994 non-null   object 
 4   complaint_time     5000 non-null   object 
 5   city               5000 non-null   object 
 6   circle             5000 non-null   object 
 7   complaint_channel  5000 non-null   object 
 8   issue_category     4710 non-null   object 
 9   issue_description  5000 non-null   object 
 10  complaint_status   4541 non-null   object 
 11  sentiment_tag      5000 non-null   object 
 12  network_type       5000 non-null   object 
 13  device_type        5000 non-null   object 
 14  amount_impacted    4571 non-null   float64
dtypes: float64(1), object(14)
memory usage: 586.1+ KB
Unique Complaints: 490

,0
complaint_date,6
issue_category,290
complaint_status,459
amount_impacted,429


,count
issue_category,
network down,416
recharge failed,356
SLOW DATA,326
No Signal,321
call drop,312
internet not working,299
NaN,290
network outage,267
service request pending,265


## Milestone 02 · Onboard the XLSX network source and JSON reference collections

**From the team:** Network engineering provides Excel incident logs, while customer, site, recharge, and service-request histories are organized as distinct collections inside a single JSON reference file.

**Your responsibility:** Read the workbook and JSON; materialize five independent source DataFrames and policy notes. Do not merge repeated recharge or request records prematurely.

**Required evidence:** `customers_df`, `cell_sites_df`, `network_df`, `recharge_df`, `requests_df`, `operations_notes`, and source row counts.

**Reasoning:** These sources have different structures: JSON reference collections, an XLSX network log, and one-to-many event records. Loading them separately and checking their shapes and keys establishes the relationships needed for safe integration.

In [3]:
# Milestone 02 — Onboard network and customer sources
# Task 1: Parse careops_reference.json and inspect its named collections and operational notes.
# Task 2: Build customers_df and cell_sites_df from their distinct JSON collections.
# Task 3: Load the XLSX source into network_df; inspect network-report identifiers and severity.
# Task 4: Convert JSON recharge_failures and service_requests collections into recharge_df and requests_df.
# Task 5: Preserve operations_notes as a string, and display source record counts and sample rows.
# Required output: customers_df, cell_sites_df, network_df, recharge_df, requests_df, operations_notes.

# ============================================================
# MILESTONE 02: ONBOARD NETWORK AND CUSTOMER SOURCES
# ============================================================

# TASK 1: Load the JSON reference file
json_path = DATA_DIR / 'careops_reference.json'

with open(json_path, 'r', encoding='utf-8') as file:
    careops_data = json.load(file)

print("JSON file loaded successfully!")

# Inspect available JSON collections
print("\nAvailable JSON keys:")
print(careops_data.keys())


# TASK 2: Create Customers and Cell Sites DataFrames

customers_df = pd.DataFrame(careops_data['customers'])

cell_sites_df = pd.DataFrame(careops_data['cell_sites'])


# TASK 3: Load Network Reports from Excel

excel_path = DATA_DIR / 'network_issue_reports.xlsx'

network_df = pd.read_excel(excel_path)

print("\nExcel file loaded successfully!")


# TASK 4: Create Recharge and Service Request DataFrames

recharge_df = pd.DataFrame(careops_data['recharge_failures'])

requests_df = pd.DataFrame(careops_data['service_requests'])


# TASK 5: Preserve Operational Notes

operations_notes = careops_data['network_operations_notes']

# Ensure operational notes remain a string
operations_notes = str(operations_notes)


# ============================================================
# DISPLAY SOURCE RECORD COUNTS
# ============================================================

source_counts = pd.DataFrame({
    'Source': [
        'Customers',
        'Cell Sites',
        'Network Reports',
        'Recharge Failures',
        'Service Requests'
    ],
    'Total Records': [
        len(customers_df),
        len(cell_sites_df),
        len(network_df),
        len(recharge_df),
        len(requests_df)
    ]
})

print("\n========== SOURCE RECORD COUNTS ==========")
display(source_counts)


# ============================================================
# DISPLAY DATASET SHAPES
# ============================================================

print("\n========== DATASET SHAPES ==========")

print("Customers:", customers_df.shape)
print("Cell Sites:", cell_sites_df.shape)
print("Network Reports:", network_df.shape)
print("Recharge Failures:", recharge_df.shape)
print("Service Requests:", requests_df.shape)


# ============================================================
# DISPLAY SAMPLE RECORDS
# ============================================================

print("\n========== CUSTOMERS SAMPLE ==========")
display(customers_df.head())

print("\n========== CELL SITES SAMPLE ==========")
display(cell_sites_df.head())

print("\n========== NETWORK REPORTS SAMPLE ==========")
display(network_df.head())

print("\n========== RECHARGE FAILURES SAMPLE ==========")
display(recharge_df.head())

print("\n========== SERVICE REQUESTS SAMPLE ==========")
display(requests_df.head())


# ============================================================
# INSPECT NETWORK SEVERITY
# ============================================================

print("\n========== NETWORK SEVERITY ==========")
display(network_df['severity'].value_counts(dropna=False))


# ============================================================
# INSPECT OPERATIONAL NOTES
# ============================================================

print("\n========== NETWORK OPERATIONS NOTES ==========")
print("Data type:", type(operations_notes).__name__)
print("Total characters:", len(operations_notes))
print("\nFirst 1000 characters:\n")
print(operations_notes[:1000])


# ============================================================
# MILESTONE 02 COMPLETED
# ============================================================

print("\nMilestone 02: All five source DataFrames created successfully!")


JSON file loaded successfully!

Available JSON keys:
dict_keys(['dataset_description', 'customers', 'cell_sites', 'recharge_failures', 'service_requests', 'network_operations_notes'])

Excel file loaded successfully!

========== SOURCE RECORD COUNTS ==========


,Source,Total Records
0,Customers,1200
1,Cell Sites,1000
2,Network Reports,1300
3,Recharge Failures,1500
4,Service Requests,1500



========== DATASET SHAPES ==========
Customers: (1200, 9)
Cell Sites: (1000, 9)
Network Reports: (1300, 10)
Recharge Failures: (1500, 9)
Service Requests: (1500, 10)

========== CUSTOMERS SAMPLE ==========


,customer_id,customer_name,customer_segment,plan_type,home_city,circle,account_age_days,preferred_service,monthly_recharge_value
0,CUST00001,Krishna Mehta,Prepaid,4G Value,Indore,Madhya Pradesh,581,Mobile Data,1499
1,CUST00002,Vivaan Sharma,Prepaid,5G Unlimited,Delhi,Delhi NCR,905,Voice,1499
2,CUST00003,Reyansh Kulkarni,Prepaid,4G Value,Lucknow,UP East,1849,Roaming,399
3,CUST00004,Sai Kulkarni,Postpaid,Fiber Silver,Mumbai,Mumbai,1148,Voice,299
4,CUST00005,Myra Gupta,Prepaid,Fiber Silver,Kolkata,West Bengal,1418,Roaming,399



========== CELL SITES SAMPLE ==========


...


========== SERVICE REQUESTS SAMPLE ==========


,service_request_id,complaint_id,customer_id,request_created_date,request_closed_date,request_type,service_status,assigned_team,resolution_time_hours,escalation_flag
0,SR000001,CMP004694,CUST00217,2026-04-06 02:00,,Recharge Support,Escalated,Recharge Ops,,Y
1,SR000002,CMP001907,CUST00813,2026-03-15 00:00,,Recharge Support,Open,Fiber Field Team,56,yes
2,SR000003,CMP003243,CUST00377,2026-05-07 21:00,2026-05-08 08:00,Billing Investigation,Closed,Care Escalation Desk,,N
3,SR000004,CMP002745,CUST01194,2026-05-13 23:00,2026-05-14 08:00,Fiber Technician,Closed,Retail Ops,,No
4,SR000005,CMP002664,CUST00471,2026-04-10 23:00,2026-04-13 06:00,Network Visit,Resolved,Retail Ops,None,N



========== NETWORK SEVERITY ==========


,count
severity,
Low,201
HIGH,196
critical,188
High,185
Critical,183
Medium,174
major outage,173



========== NETWORK OPERATIONS NOTES ==========
Data type: str
Total characters: 8734

First 1000 characters:

001. Circle Ops observed repeated call drop complaints near Ahmedabad commuter corridor.
002. Manager asks developer to validate orphan complaint and service request links before final reporting.
003. NOC recommends checking high capacity utilization sites in Kolkata.
004. Circle Ops observed repeated call drop complaints near Surat commuter corridor.
005. Circle Ops observed repeated call drop complaints near Kolkata commuter corridor.
006. Recharge Ops flagged delayed refund batches for UPI payment mode in Bengaluru.
007. Circle Ops observed repeated call drop complaints near Jaipur commuter corridor.
008. Recharge Ops flagged delayed refund batches for UPI payment mode in Lucknow.
009. Circle Ops observed repeated call drop complaints near Surat commuter corridor.
010. PM reminder: final notebook must show output tables, GUI evidence, and export confirmation.
011. Manager a

## Milestone 03 · Validate and standardize complaint records

**From the team:** Quality assurance warns that missing issue labels, mixed-case statuses, and invalid timestamps make complaint-age comparisons unreliable.

**Your responsibility:** Deduplicate only by complaint_id; standardize categories and statuses; parse dates and monetary fields. Record missing customer/site references and unparseable dates without erasing valid multi-complaint histories.

**Handover evidence:** Create `complaints_clean`, `quality_summary` and normalized `issue_group` with complaint-level uniqueness.

**Reasoning:** Duplicate complaint IDs can inflate workload, while inconsistent labels split the same issue into multiple categories. Standardizing categories and statuses improves grouping; parsing dates and amounts makes later analysis possible. Missing and invalid values are recorded rather than hidden.

In [4]:
# Milestone 03 — Validate and standardize complaint records
# Task 1: Deduplicate only by complaint_id; standardize categories and statuses; parse dates and monetary fields. Record missing customer/site references and unparseable dates without erasing valid multi-complaint histories.
# Task 2: Produce this required evidence: Create `complaints_clean`, `quality_summary` and normalized `issue_group` with complaint-level uniqueness.
# Implement your code below and display the requested outputs.

# ============================================================
# MILESTONE 03: VALIDATE AND STANDARDIZE COMPLAINT RECORDS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# TASK 1: CREATE A WORKING COPY
# ------------------------------------------------------------

complaints_clean = complaints_df.copy()

print("Original records:", len(complaints_clean))


# ------------------------------------------------------------
# TASK 2: IDENTIFY AND REMOVE DUPLICATE COMPLAINTS
# Deduplicate ONLY using complaint_id
# ------------------------------------------------------------

duplicate_rows = complaints_clean['complaint_id'].duplicated().sum()

duplicate_ids = complaints_clean.loc[
    complaints_clean['complaint_id'].duplicated(keep=False),
    'complaint_id'
].nunique()

print("\nDuplicate complaint rows:", duplicate_rows)
print("Distinct duplicated complaint IDs:", duplicate_ids)

# Keep the first record for each complaint_id
complaints_clean = complaints_clean.drop_duplicates(
    subset=['complaint_id'],
    keep='first'
).copy()

print("Records after deduplication:", len(complaints_clean))


# ------------------------------------------------------------
# TASK 3: STANDARDIZE ISSUE CATEGORIES
# ------------------------------------------------------------

# Remove unnecessary spaces and normalize text
complaints_clean['issue_category'] = (
    complaints_clean['issue_category']
    .astype('string')
    .str.strip()
    .str.lower()
)

# Replace empty strings with missing values
complaints_clean['issue_category'] = (
    complaints_clean['issue_category'].replace('', pd.NA)
)

# Standardize inconsistent category names
category_mapping = {

    'network down': 'Network Down',
    'network outage': 'Network Down',
    'tower issue': 'Network Down',
    'fiber outage': 'Network Down',

    'no signal': 'Signal Issue',
    'weak signal': 'Signal Issue',

    'slow data': 'Slow Internet',
    'data speed low': 'Slow Internet',
    'internet not working': 'Internet Not Working',

    'call drop': 'Call Drop',

    'recharge failed': 'Recharge Failure',
    'payment deducted recharge not done': 'Recharge Failure',
    'refund not received': 'Refund Issue',

    'service request pending': 'Service Request',
    'sim activation pending': 'Service Request',

    'billing issue': 'Billing Issue',

    'complaint unresolved': 'Unresolved Complaint'
}

complaints_clean['issue_category'] = (
    complaints_clean['issue_category']
    .map(category_mapping)
    .fillna('Unknown')
)


# ------------------------------------------------------------
# TASK 4: CREATE NORMALIZED ISSUE GROUP
# ------------------------------------------------------------

issue_group_mapping = {

    'Network Down': 'Network',
    'Signal Issue': 'Network',
    'Call Drop': 'Network',

    'Slow Internet': 'Data',
    'Internet Not Working': 'Data',

    'Recharge Failure': 'Recharge',
    'Refund Issue': 'Recharge',

    'Service Request': 'Service Request',
    'Billing Issue': 'Billing',

    'Unresolved Complaint': 'General',
    'Unknown': 'Unknown'
}

complaints_clean['issue_group'] = (
    complaints_clean['issue_category']
    .map(issue_group_mapping)
    .fillna('Unknown')
)


# ------------------------------------------------------------
# TASK 5: STANDARDIZE COMPLAINT STATUS
# ------------------------------------------------------------

complaints_clean['complaint_status'] = (
    complaints_clean['complaint_status']
    .astype('string')
    .str.strip()
    .str.lower()
)

# Convert blank strings to missing values
complaints_clean['complaint_status'] = (
    complaints_clean['complaint_status'].replace('', pd.NA)
)

status_mapping = {

    'open': 'Open',
    'in progress': 'In Progress',
    'in-progress': 'In Progress',
    'pending': 'Pending',
    'pending review': 'Pending Review',
    'escalated': 'Escalated',
    'resolved': 'Resolved',
    'closed': 'Closed'
}

complaints_clean['complaint_status'] = (
    complaints_clean['complaint_status']
    .map(status_mapping)
    .fillna('Unknown')
)


# ------------------------------------------------------------
# TASK 6: CONVERT COMPLAINT DATE INTO DATETIME
# ------------------------------------------------------------

complaints_clean['complaint_date'] = pd.to_datetime(
    complaints_clean['complaint_date'],
    errors='coerce'
)

# Identify missing or invalid dates
complaints_clean['invalid_complaint_date'] = (
    complaints_clean['complaint_date'].isna()
)

print("\nInvalid or missing complaint dates:",
      complaints_clean['invalid_complaint_date'].sum())


# ------------------------------------------------------------
# TASK 7: CONVERT AMOUNT IMPACTED INTO NUMERIC
# ------------------------------------------------------------

complaints_clean['amount_impacted'] = pd.to_numeric(
    complaints_clean['amount_impacted'],
    errors='coerce'
)


# ------------------------------------------------------------
# TASK 8: VALIDATE CUSTOMER AND CELL SITE REFERENCES
# ------------------------------------------------------------

valid_customer_ids = set(customers_df['customer_id'].dropna())

valid_site_ids = set(cell_sites_df['cell_site_id'].dropna())

# Identify customer IDs not present in customers reference table
complaints_clean['missing_customer_reference'] = (
    complaints_clean['customer_id'].isna()
    | ~complaints_clean['customer_id'].isin(valid_customer_ids)
)

# Identify cell site IDs not present in cell sites reference table
complaints_clean['missing_site_reference'] = (
    complaints_clean['cell_site_id'].isna()
    | ~complaints_clean['cell_site_id'].isin(valid_site_ids)
)


# ------------------------------------------------------------
# TASK 9: CREATE DATA QUALITY SUMMARY
# ------------------------------------------------------------

quality_summary = pd.DataFrame({

    'Quality Check': [

        'Original complaint records',

        'Duplicate complaint rows removed',

        'Distinct duplicated complaint IDs',

        'Clean complaint records',

        'Missing issue categories',

        'Unknown issue categories after cleaning',

        'Missing complaint statuses',

        'Unknown complaint statuses after cleaning',

        'Missing complaint dates',

        'Invalid or missing complaint dates',

        'Missing customer references',

        'Missing cell site references',

        'Missing amount impacted'

    ],

    'Count': [

        len(complaints_df),

        duplicate_rows,

        duplicate_ids,

        len(complaints_clean),

        complaints_df['issue_category'].isna().sum(),

        (complaints_clean['issue_category'] == 'Unknown').sum(),

        complaints_df['complaint_status'].isna().sum(),

        (complaints_clean['complaint_status'] == 'Unknown').sum(),

        complaints_df['complaint_date'].isna().sum(),

        complaints_clean['invalid_complaint_date'].sum(),

        complaints_clean['missing_customer_reference'].sum(),

        complaints_clean['missing_site_reference'].sum(),

        complaints_clean['amount_impacted'].isna().sum()

    ]

})

print("\n========== DATA QUALITY SUMMARY ==========")
display(quality_summary)


# ------------------------------------------------------------
# TASK 10: VERIFY CLEANED DATA
# ------------------------------------------------------------

print("\n========== CLEANED DATA INFORMATION ==========")

print("Clean dataset shape:", complaints_clean.shape)

print("Unique complaint IDs:",
      complaints_clean['complaint_id'].nunique())

print("Duplicate complaint IDs:",
      complaints_clean['complaint_id'].duplicated().sum())

print("Complaint date data type:",
      complaints_clean['complaint_date'].dtype)

print("Amount impacted data type:",
      complaints_clean['amount_impacted'].dtype)


# ------------------------------------------------------------
# TASK 11: DISPLAY CLEANED CATEGORY DISTRIBUTION
# ------------------------------------------------------------

print("\n========== STANDARDIZED ISSUE CATEGORIES ==========")
display(
    complaints_clean['issue_category']
    .value_counts(dropna=False)
    .to_frame('Complaint Count')
)

print("\n========== NORMALIZED ISSUE GROUP ==========")
display(
    complaints_clean['issue_group']
    .value_counts(dropna=False)
    .to_frame('Complaint Count')
)


# ------------------------------------------------------------
# TASK 12: DISPLAY STANDARDIZED COMPLAINT STATUS
# ------------------------------------------------------------

print("\n========== STANDARDIZED COMPLAINT STATUS ==========")

display(
    complaints_clean['complaint_status']
    .value_counts(dropna=False)
    .to_frame('Complaint Count')
)


# ------------------------------------------------------------
# TASK 13: FINAL SAMPLE
# ------------------------------------------------------------

print("\n========== FINAL CLEANED COMPLAINT DATA ==========")

display(
    complaints_clean[
        [
            'complaint_id',
            'customer_id',
            'cell_site_id',
            'complaint_date',
            'issue_category',
            'issue_group',
            'complaint_status',
            'amount_impacted',
            'invalid_complaint_date',
            'missing_customer_reference',
            'missing_site_reference'
        ]
    ].head(10)
)


# ------------------------------------------------------------
# MILESTONE 03 COMPLETED
# ------------------------------------------------------------

print("\nMilestone 03 completed successfully!")

assert complaints_clean['complaint_id'].notna().all()
assert complaints_clean['complaint_id'].is_unique
assert pd.api.types.is_datetime64_any_dtype(
    complaints_clean['complaint_date']
)

print("All key validation checks passed.")


Original records: 5000

Duplicate complaint rows: 99
Distinct duplicated complaint IDs: 1
Records after deduplication: 4901

Invalid or missing complaint dates: 12

========== DATA QUALITY SUMMARY ==========


,Quality Check,Count
0,Original complaint records,5000
1,Duplicate complaint rows removed,99
2,Distinct duplicated complaint IDs,1
3,Clean complaint records,4901
4,Missing issue categories,290
5,Unknown issue categories after cleaning,545
6,Missing complaint statuses,459
7,Unknown complaint statuses after cleaning,452
8,Missing complaint dates,6
9,Invalid or missing complaint dates,12



========== CLEANED DATA INFORMATION ==========
Clean dataset shape: (4901, 19)
Unique complaint IDs: 4901
Duplicate complaint IDs: 0
Complaint date data type: datetime64[ns]
Amount impacted data type: float64

========== STANDARDIZED ISSUE CATEGORIES ==========


,Complaint Count
issue_category,
Network Down,857
Recharge Failure,837
Slow Internet,564
Signal Issue,558
Unknown,545
Service Request,453
Call Drop,304
Internet Not Working,295
Refund Issue,206



========== NORMALIZED ISSUE GROUP ==========


...


========== STANDARDIZED COMPLAINT STATUS ==========


,Complaint Count
complaint_status,
Closed,922
Open,864
In Progress,844
Pending Review,474
Escalated,453
Unknown,452
Resolved,448
Pending,444



========== FINAL CLEANED COMPLAINT DATA ==========


,complaint_id,customer_id,cell_site_id,complaint_date,issue_category,issue_group,complaint_status,amount_impacted,invalid_complaint_date,missing_customer_reference,missing_site_reference
0,CMP000001,CUST00246,SITE0878,2026-05-04,Unknown,Unknown,Open,799.0,False,False,False
1,CMP000002,CUST00793,SITE0461,2026-02-05,Unknown,Unknown,Unknown,399.0,False,False,False
2,CMP000003,CUST00931,SITE0550,2026-02-09,Unknown,Unknown,In Progress,199.0,False,False,False
3,CMP000004,CUST00043,SITE0443,2026-02-18,Internet Not Working,Data,Open,399.0,False,False,False
4,CMP000005,CUST00723,SITE0214,2026-02-01,Recharge Failure,Recharge,Closed,399.0,False,False,False
5,CMP000006,CUST00668,SITE0682,2026-04-11,Signal Issue,Network,Pending,149.0,False,False,False
6,CMP000007,CUST00745,SITE0922,2026-04-03,Service Request,Service Request,Closed,239.0,False,False,False
7,CMP000008,CUST01112,SITE0668,2026-04-05,Internet Not Working,Data,Pending Review,299.0,False,False,False
8,CMP000009,CUST01060,SITE0443,2026-05-09,Unresolved Complaint,General,Open,149.0,False,False,False
9,CMP000010,CUST01178,SITE0801,2026-02-19,Slow Internet,Data,Closed,239.0,False,False,False



Milestone 03 completed successfully!
All key validation checks passed.


## Milestone 04 · Prepare reliable network incident signals

**From the team:** Network operations wants tower status and incident severity contextualized before escalation decisions.

**Your responsibility:** Deduplicate network report IDs; normalize severity/status; aggregate distinct incidents at cell-site level; join site inventory without multiplying incidents.

**Handover evidence:** Create `network_clean`, `network_by_site`, and `sites_clean` with unique site IDs.

**Reasoning:** Multiple network reports can exist for one cell site. Aggregating incident counts and impact measures at site level before joining prevents network rows from multiplying complaint records. Severity and current status are normalized so operational signals use consistent labels.

In [5]:
# Milestone 04 — Prepare reliable network incident signals
# Task 1: Deduplicate network report IDs; normalize severity/status; aggregate distinct incidents at cell-site level; join site inventory without multiplying incidents.
# Task 2: Produce this required evidence: Create `network_clean`, `network_by_site`, and `sites_clean` with unique site IDs.
# Implement your code below and display the requested outputs.

# ============================================================
# MILESTONE 04: PREPARE RELIABLE NETWORK INCIDENT SIGNALS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# TASK 1: CREATE A WORKING COPY OF NETWORK REPORTS
# ------------------------------------------------------------

network_clean = network_df.copy()

print("Original network reports:", len(network_clean))


# ------------------------------------------------------------
# TASK 2: REMOVE DUPLICATE NETWORK REPORT IDs
# ------------------------------------------------------------

duplicate_network_rows = (
    network_clean['network_report_id'].duplicated().sum()
)

print("Duplicate network report rows:", duplicate_network_rows)

network_clean = network_clean.drop_duplicates(
    subset=['network_report_id'],
    keep='first'
).copy()

print("Network reports after deduplication:", len(network_clean))


# ------------------------------------------------------------
# TASK 3: STANDARDIZE NETWORK SEVERITY
# ------------------------------------------------------------

network_clean['severity'] = (
    network_clean['severity']
    .astype('string')
    .str.strip()
    .str.lower()
)

network_clean['severity'] = (
    network_clean['severity'].replace('', pd.NA)
)

severity_mapping = {
    'critical': 'Critical',
    'major outage': 'Critical',
    'high': 'High',
    'medium': 'Medium',
    'low': 'Low'
}

network_clean['severity'] = (
    network_clean['severity']
    .map(severity_mapping)
    .fillna('Unknown')
)


# ------------------------------------------------------------
# TASK 4: STANDARDIZE NETWORK REPORT STATUS
# ------------------------------------------------------------

network_clean['current_status'] = (
    network_clean['current_status']
    .astype('string')
    .str.strip()
    .str.lower()
)

network_clean['current_status'] = (
    network_clean['current_status'].replace('', pd.NA)
)

network_status_mapping = {
    'open': 'Open',
    'pending': 'Pending',
    'monitoring': 'Monitoring',
    'in progress': 'In Progress',
    'in-progress': 'In Progress',
    'resolved': 'Resolved',
    'closed': 'Closed'
}

network_clean['current_status'] = (
    network_clean['current_status']
    .map(network_status_mapping)
    .fillna('Unknown')
)


# ------------------------------------------------------------
# TASK 5: CONVERT REPORT DATE INTO DATETIME
# ------------------------------------------------------------

network_clean['report_date'] = pd.to_datetime(
    network_clean['report_date'],
    errors='coerce'
)

print("\nInvalid or missing network report dates:",
      network_clean['report_date'].isna().sum())


# ------------------------------------------------------------
# TASK 6: CONVERT NUMERIC COLUMNS
# ------------------------------------------------------------

numeric_columns = [
    'affected_customers',
    'avg_downtime_minutes'
]

for col in numeric_columns:
    network_clean[col] = pd.to_numeric(
        network_clean[col],
        errors='coerce'
    )


# ------------------------------------------------------------
# TASK 7: CLEAN CELL SITE INVENTORY
# ------------------------------------------------------------

sites_clean = cell_sites_df.copy()

print("\nOriginal cell site records:", len(sites_clean))

duplicate_site_rows = (
    sites_clean['cell_site_id'].duplicated().sum()
)

print("Duplicate cell site IDs:", duplicate_site_rows)

# Keep one record per cell site
sites_clean = sites_clean.drop_duplicates(
    subset=['cell_site_id'],
    keep='first'
).copy()

# Normalize site status
sites_clean['site_status'] = (
    sites_clean['site_status']
    .astype('string')
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

sites_clean['site_status'] = (
    sites_clean['site_status'].replace('', pd.NA)
)

# Convert maintenance date
sites_clean['last_maintenance_date'] = pd.to_datetime(
    sites_clean['last_maintenance_date'],
    errors='coerce'
)

# Convert site performance metrics into numeric values
site_numeric_columns = [
    'avg_signal_strength_dbm',
    'capacity_utilization_pct'
]

for col in site_numeric_columns:
    sites_clean[col] = pd.to_numeric(
        sites_clean[col],
        errors='coerce'
    )


# ------------------------------------------------------------
# TASK 8: IDENTIFY SEVERE NETWORK REPORTS
# ------------------------------------------------------------

# Critical, High and major outage (standardized as Critical)
severe_levels = ['Critical', 'High']

network_clean['is_severe_network_report'] = (
    network_clean['severity'].isin(severe_levels)
)

# Identify severe reports that are still open or being monitored
open_statuses = [
    'Open',
    'Pending',
    'Monitoring',
    'In Progress'
]

network_clean['is_open_severe_network_report'] = (
    network_clean['is_severe_network_report']
    & network_clean['current_status'].isin(open_statuses)
)


# ------------------------------------------------------------
# TASK 9: AGGREGATE NETWORK REPORTS AT CELL SITE LEVEL
# ------------------------------------------------------------

network_by_site = (
    network_clean
    .dropna(subset=['cell_site_id'])
    .groupby('cell_site_id')
    .agg(
        network_report_count=(
            'network_report_id', 'nunique'
        ),

        severe_network_report_count=(
            'is_severe_network_report', 'sum'
        ),

        open_severe_network_report_count=(
            'is_open_severe_network_report', 'sum'
        ),

        total_affected_customers=(
            'affected_customers', 'sum'
        ),

        avg_downtime_minutes=(
            'avg_downtime_minutes', 'mean'
        )
    )
    .reset_index()
)

print("\nNetwork aggregation completed!")


# ------------------------------------------------------------
# TASK 10: JOIN SITE INVENTORY WITHOUT MULTIPLYING INCIDENTS
# ------------------------------------------------------------

# Both tables must have unique cell_site_id values.
# This is a one-to-one site-level validation join.

network_by_site = network_by_site.merge(
    sites_clean[['cell_site_id']],
    on='cell_site_id',
    how='left',
    validate='one_to_one',
    indicator=True
)

network_by_site['site_inventory_match'] = (
    network_by_site['_merge'] == 'both'
)

network_by_site.drop(
    columns=['_merge'],
    inplace=True
)


# ------------------------------------------------------------
# TASK 11: CREATE NETWORK QUALITY SUMMARY
# ------------------------------------------------------------

network_quality_summary = pd.DataFrame({

    'Quality Check': [
        'Original network reports',
        'Duplicate network report rows',
        'Clean network reports',
        'Unique cell sites in inventory',
        'Sites with network reports',
        'Network reports with missing site ID',
        'Network reports with invalid dates',
        'Unknown severity records',
        'Unknown network status records',
        'Network sites matching inventory',
        'Network sites not matching inventory'
    ],

    'Count': [
        len(network_df),
        duplicate_network_rows,
        len(network_clean),
        sites_clean['cell_site_id'].nunique(),
        len(network_by_site),
        network_clean['cell_site_id'].isna().sum(),
        network_clean['report_date'].isna().sum(),
        (network_clean['severity'] == 'Unknown').sum(),
        (network_clean['current_status'] == 'Unknown').sum(),
        network_by_site['site_inventory_match'].sum(),
        (~network_by_site['site_inventory_match']).sum()
    ]

})

print("\n========== NETWORK QUALITY SUMMARY ==========")
display(network_quality_summary)


# ------------------------------------------------------------
# TASK 12: DISPLAY NETWORK SEVERITY DISTRIBUTION
# ------------------------------------------------------------

print("\n========== STANDARDIZED NETWORK SEVERITY ==========")

display(
    network_clean['severity']
    .value_counts(dropna=False)
    .to_frame('Report Count')
)


# ------------------------------------------------------------
# TASK 13: DISPLAY NETWORK STATUS DISTRIBUTION
# ------------------------------------------------------------

print("\n========== STANDARDIZED NETWORK STATUS ==========")

display(
    network_clean['current_status']
    .value_counts(dropna=False)
    .to_frame('Report Count')
)


# ------------------------------------------------------------
# TASK 14: DISPLAY SITE STATUS DISTRIBUTION
# ------------------------------------------------------------

print("\n========== CELL SITE STATUS ==========")

display(
    sites_clean['site_status']
    .value_counts(dropna=False)
    .to_frame('Site Count')
)


# ------------------------------------------------------------
# TASK 15: DISPLAY NETWORK AGGREGATION
# ------------------------------------------------------------

print("\n========== NETWORK INCIDENTS BY SITE ==========")

display(network_by_site.head(10))


# ------------------------------------------------------------
# TASK 16: VALIDATION CHECKS
# ------------------------------------------------------------

assert network_clean['network_report_id'].is_unique

assert sites_clean['cell_site_id'].notna().all()

assert sites_clean['cell_site_id'].is_unique

assert network_by_site['cell_site_id'].notna().all()

assert network_by_site['cell_site_id'].is_unique

expected_network_count = (
    network_clean['cell_site_id'].notna().sum()
)

actual_network_count = network_by_site[
    'network_report_count'
].sum()

assert actual_network_count == expected_network_count

print("\nNetwork report count before aggregation:",
      expected_network_count)

print("Network report count after aggregation:",
      actual_network_count)

print("\nMilestone 04 completed successfully!")
print("All network validation checks passed.")



Original network reports: 1300
Duplicate network report rows: 0
Network reports after deduplication: 1300

Invalid or missing network report dates: 8

Original cell site records: 1000
Duplicate cell site IDs: 0

Network aggregation completed!

========== NETWORK QUALITY SUMMARY ==========


,Quality Check,Count
0,Original network reports,1300
1,Duplicate network report rows,0
2,Clean network reports,1300
3,Unique cell sites in inventory,1000
4,Sites with network reports,721
5,Network reports with missing site ID,0
6,Network reports with invalid dates,8
7,Unknown severity records,0
8,Unknown network status records,222
9,Network sites matching inventory,704



========== STANDARDIZED NETWORK SEVERITY ==========


,Report Count
severity,
Critical,544
High,381
Low,201
Medium,174



========== STANDARDIZED NETWORK STATUS ==========


...


========== CELL SITE STATUS ==========


,Site Count
site_status,
Active,496
Planned Upgrade,187
Under Maintenance,173
Congested,144



========== NETWORK INCIDENTS BY SITE ==========


,cell_site_id,network_report_count,severe_network_report_count,open_severe_network_report_count,total_affected_customers,avg_downtime_minutes,site_inventory_match
0,SITE0005,1,1,1,1027,516.0,True
1,SITE0008,2,1,1,1217,445.0,True
2,SITE0010,2,1,0,1670,375.0,True
3,SITE0011,2,2,2,1432,429.5,True
4,SITE0013,3,1,0,4447,394.0,True
5,SITE0014,1,0,0,678,585.0,True
6,SITE0015,4,2,2,4120,458.0,True
7,SITE0017,1,0,0,454,420.0,True
8,SITE0018,1,1,0,1594,462.0,True
9,SITE0019,2,0,0,1632,369.0,True



Network report count before aggregation: 1300
Network report count after aggregation: 1300

Milestone 04 completed successfully!
All network validation checks passed.


## Milestone 05 · Audit recharge failures and refund risk

**From the team:** Recharge Support needs to spot unresolved money-impacting cases and failed payments linked to complaints.

**Your responsibility:** Normalize recharge status, parse amounts and dates, flag failed/pending outcomes and pending refunds, and aggregate by complaint ID. Do not discard valid multiple recharge events.

**Handover evidence:** Create `recharge_clean`, `recharge_by_complaint` and readable risk counts.

**Reasoning:** A complaint may have multiple recharge attempts, so valid events must be retained and summarized at complaint level. Failed/pending payment and refund indicators expose follow-up risk; missing amounts and unknown statuses remain data-quality limitations, not proof of a successful or failed transaction.

In [6]:
# Milestone 05 — Audit recharge failures and refund risk
# Task 1: Normalize recharge status, parse amounts and dates, flag failed/pending outcomes and pending refunds, and aggregate by complaint ID. Do not discard valid multiple recharge events.
# Task 2: Produce this required evidence: Create `recharge_clean`, `recharge_by_complaint` and readable risk counts.
# Implement your code below and display the requested outputs.
# ============================================================
# MILESTONE 5: RECHARGE DATA CLEANING & RISK ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Create a working copy
# ------------------------------------------------------------

recharge_clean = recharge_df.copy()

print("Original Recharge Dataset Shape:", recharge_clean.shape)


# ------------------------------------------------------------
# 2. Remove duplicate recharge records
# ------------------------------------------------------------

recharge_duplicate_count = recharge_clean["recharge_id"].duplicated().sum()

recharge_clean = recharge_clean.drop_duplicates(
    subset=["recharge_id"],
    keep="first"
).copy()

print("Duplicate Recharge Records Removed:", recharge_duplicate_count)


# ------------------------------------------------------------
# 3. Standardize recharge status
# ------------------------------------------------------------

recharge_clean["recharge_status"] = (
    recharge_clean["recharge_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

recharge_status_mapping = {
    "success": "Successful",
    "successful": "Successful",
    "completed": "Successful",
    "failed": "Failed",
    "failure": "Failed",
    "failed payment": "Failed",
    "pending": "Pending",
    "recharge pending": "Pending",
    "processing": "Pending"
}

recharge_clean["recharge_status_clean"] = (
    recharge_clean["recharge_status"]
    .map(recharge_status_mapping)
    .fillna("Unknown")
)


# ------------------------------------------------------------
# 4. Standardize refund status
# ------------------------------------------------------------

recharge_clean["refund_status"] = (
    recharge_clean["refund_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

refund_status_mapping = {
    "completed": "Completed",
    "success": "Completed",
    "successful": "Completed",
    "pending": "Pending",
    "pending refund": "Pending",
    "processing": "Pending",
    "failed": "Failed",
    "failure": "Failed",
    "not initiated": "Not Initiated"
}

recharge_clean["refund_status_clean"] = (
    recharge_clean["refund_status"]
    .map(refund_status_mapping)
    .fillna("Unknown")
)


# ------------------------------------------------------------
# 5. Convert date and amount columns
# ------------------------------------------------------------

recharge_clean["recharge_date"] = pd.to_datetime(
    recharge_clean["recharge_date"],
    errors="coerce"
)

recharge_clean["refund_completed_date"] = pd.to_datetime(
    recharge_clean["refund_completed_date"],
    errors="coerce"
)

recharge_clean["recharge_amount"] = pd.to_numeric(
    recharge_clean["recharge_amount"],
    errors="coerce"
)


# ------------------------------------------------------------
# 6. Create recharge risk flags
# ------------------------------------------------------------

recharge_clean["is_failed_recharge"] = (
    recharge_clean["recharge_status_clean"] == "Failed"
)

recharge_clean["is_pending_recharge"] = (
    recharge_clean["recharge_status_clean"] == "Pending"
)

recharge_clean["is_pending_refund"] = (
    recharge_clean["refund_status_clean"] == "Pending"
)

recharge_clean["is_failed_refund"] = (
    recharge_clean["refund_status_clean"] == "Failed"
)

# A recharge record is considered risky if its recharge
# failed/pending or its refund is pending/failed.

recharge_clean["recharge_risk_flag"] = (
    recharge_clean["is_failed_recharge"]
    | recharge_clean["is_pending_recharge"]
    | recharge_clean["is_pending_refund"]
    | recharge_clean["is_failed_refund"]
)


# ------------------------------------------------------------
# 7. Create a readable quality summary
# ------------------------------------------------------------

recharge_quality_summary = pd.DataFrame({
    "Metric": [
        "Original recharge records",
        "Duplicate recharge records removed",
        "Clean recharge records",
        "Missing complaint IDs",
        "Missing recharge amounts",
        "Invalid recharge dates",
        "Failed recharge records",
        "Pending recharge records",
        "Pending refund records",
        "Failed refund records",
        "Total recharge risk records"
    ],

    "Value": [
        len(recharge_df),
        recharge_duplicate_count,
        len(recharge_clean),
        recharge_clean["complaint_id"].isna().sum(),
        recharge_clean["recharge_amount"].isna().sum(),
        recharge_clean["recharge_date"].isna().sum(),
        recharge_clean["is_failed_recharge"].sum(),
        recharge_clean["is_pending_recharge"].sum(),
        recharge_clean["is_pending_refund"].sum(),
        recharge_clean["is_failed_refund"].sum(),
        recharge_clean["recharge_risk_flag"].sum()
    ]
})


# ------------------------------------------------------------
# 8. Aggregate recharge data by complaint_id
# ------------------------------------------------------------

# Exclude records without a complaint ID from complaint-level
# aggregation. They remain available in recharge_clean.

recharge_valid = recharge_clean.dropna(
    subset=["complaint_id"]
).copy()

recharge_by_complaint = (
    recharge_valid
    .groupby("complaint_id")
    .agg(
        recharge_count=("recharge_id", "nunique"),

        failed_recharge_count=("is_failed_recharge", "sum"),

        pending_recharge_count=("is_pending_recharge", "sum"),

        pending_refund_count=("is_pending_refund", "sum"),

        failed_refund_count=("is_failed_refund", "sum"),

        recharge_risk_count=("recharge_risk_flag", "sum"),

        total_recharge_amount=("recharge_amount", "sum"),

        avg_recharge_amount=("recharge_amount", "mean"),

        latest_recharge_date=("recharge_date", "max")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 9. Create aggregation quality checks
# ------------------------------------------------------------

print("\n========== RECHARGE QUALITY SUMMARY ==========")
display(recharge_quality_summary)

print("\n========== RECHARGE STATUS DISTRIBUTION ==========")
display(
    recharge_clean["recharge_status_clean"]
    .value_counts(dropna=False)
    .rename_axis("Recharge Status")
    .reset_index(name="Record Count")
)

print("\n========== REFUND STATUS DISTRIBUTION ==========")
display(
    recharge_clean["refund_status_clean"]
    .value_counts(dropna=False)
    .rename_axis("Refund Status")
    .reset_index(name="Record Count")
)


# ------------------------------------------------------------
# 10. Display complaint-level recharge summary
# ------------------------------------------------------------

print("\n========== RECHARGE BY COMPLAINT ==========")

display(recharge_by_complaint.head(10))

print("\nRecharge Clean Shape:", recharge_clean.shape)
print("Recharge Aggregated Shape:", recharge_by_complaint.shape)

print(
    "Unique Complaint IDs:",
    recharge_by_complaint["complaint_id"].nunique()
)


# ------------------------------------------------------------
# 11. Validation checks
# ------------------------------------------------------------

assert recharge_clean["recharge_id"].is_unique, (
    "Duplicate recharge IDs still exist."
)

assert recharge_by_complaint["complaint_id"].is_unique, (
    "Duplicate complaint IDs found in recharge aggregation."
)

assert (
    recharge_by_complaint["recharge_count"].sum()
    == recharge_valid["recharge_id"].nunique()
), "Recharge aggregation count does not reconcile."

print("\nMilestone 5 validation completed successfully!")

Original Recharge Dataset Shape: (1500, 9)
Duplicate Recharge Records Removed: 0

========== RECHARGE QUALITY SUMMARY ==========


,Metric,Value
0,Original recharge records,1500
1,Duplicate recharge records removed,0
2,Clean recharge records,1500
3,Missing complaint IDs,0
4,Missing recharge amounts,389
5,Invalid recharge dates,10
6,Failed recharge records,641
7,Pending recharge records,217
8,Pending refund records,514
9,Failed refund records,244



========== RECHARGE STATUS DISTRIBUTION ==========


,Recharge Status,Record Count
0,Unknown,642
1,Failed,641
2,Pending,217



========== REFUND STATUS DISTRIBUTION ==========


,Refund Status,Record Count
0,Pending,514
1,Not Initiated,279
2,Failed,244
3,Completed,244
4,Unknown,219



========== RECHARGE BY COMPLAINT ==========


,complaint_id,recharge_count,failed_recharge_count,pending_recharge_count,pending_refund_count,failed_refund_count,recharge_risk_count,total_recharge_amount,avg_recharge_amount,latest_recharge_date
0,CMP000001,1,1,0,0,0,1,0.0,NaN,2026-04-28
1,CMP000003,1,0,0,1,0,1,999.0,999.0,2026-03-10
2,CMP000007,1,0,1,1,0,1,599.0,599.0,2026-04-11
3,CMP000009,1,0,1,0,0,1,149.0,149.0,2026-05-20
4,CMP000010,1,0,1,0,0,1,799.0,799.0,2026-03-16
5,CMP000015,1,0,1,1,0,1,599.0,599.0,2026-02-13
6,CMP000020,1,1,0,1,0,1,399.0,399.0,2026-02-08
7,CMP000021,1,1,0,1,0,1,0.0,NaN,2026-05-21
8,CMP000022,1,1,0,0,0,1,-199.0,-199.0,2026-04-15
9,CMP000023,1,1,0,0,0,1,0.0,NaN,2026-04-08



Recharge Clean Shape: (1500, 16)
Recharge Aggregated Shape: (1257, 10)
Unique Complaint IDs: 1257

Milestone 5 validation completed successfully!


## Milestone 06 · Detect service request and SLA follow-ups

**From the team:** The Service Desk wants unresolved or escalated requests visible on the original complaint without introducing duplicate complaint rows.

**Your responsibility:** Normalize service request statuses, parse request dates and resolution hours, distinguish still-open from closed cases, and aggregate by complaint before merging.

**Handover evidence:** Create `requests_clean`, `requests_by_complaint`; show unresolved and escalated counts and orphan references.

**Reasoning:** Service requests are also one-to-many. Standardizing status, escalation flags, dates, and resolution hours makes follow-up workload measurable. Aggregating by complaint before merging preserves the complaint-level grain. The 24-hour review threshold is an explicit project assumption.

In [7]:
# Milestone 06 — Detect service request and SLA follow-ups
# Task 1: Normalize service request statuses, parse request dates and resolution hours, distinguish still-open from closed cases, and aggregate by complaint before merging.
# Task 2: Produce this required evidence: Create `requests_clean`, `requests_by_complaint`; show unresolved and escalated counts and orphan references.
# Implement your code below and display the requested outputs.
# ============================================================
# MILESTONE 6: SERVICE REQUEST & SLA FOLLOW-UP ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Create a working copy
# ------------------------------------------------------------

requests_clean = requests_df.copy()

print("Original Service Request Shape:", requests_clean.shape)


# ------------------------------------------------------------
# 2. Remove duplicate service request records
# ------------------------------------------------------------

request_duplicate_count = requests_clean[
    "service_request_id"
].duplicated().sum()

requests_clean = requests_clean.drop_duplicates(
    subset=["service_request_id"],
    keep="first"
).copy()

print("Duplicate Service Requests Removed:", request_duplicate_count)


# ------------------------------------------------------------
# 3. Standardize service request status
# ------------------------------------------------------------

requests_clean["service_status"] = (
    requests_clean["service_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

service_status_mapping = {
    "open": "Open",
    "pending": "Pending",
    "in progress": "In Progress",
    "in-progress": "In Progress",
    "escalated": "Escalated",
    "closed": "Closed",
    "resolved": "Resolved",
    "complete": "Resolved",
    "completed": "Resolved"
}

requests_clean["service_status_clean"] = (
    requests_clean["service_status"]
    .map(service_status_mapping)
    .fillna("Unknown")
)


# ------------------------------------------------------------
# 4. Parse request dates
# ------------------------------------------------------------

requests_clean["request_created_date"] = pd.to_datetime(
    requests_clean["request_created_date"],
    errors="coerce"
)

requests_clean["request_closed_date"] = pd.to_datetime(
    requests_clean["request_closed_date"],
    errors="coerce"
)


# ------------------------------------------------------------
# 5. Convert resolution time into numeric format
# ------------------------------------------------------------

requests_clean["resolution_time_hours"] = pd.to_numeric(
    requests_clean["resolution_time_hours"],
    errors="coerce"
)


# ------------------------------------------------------------
# 6. Standardize escalation flag
# ------------------------------------------------------------

requests_clean["escalation_flag"] = (
    requests_clean["escalation_flag"]
    .astype("string")
    .str.strip()
    .str.lower()
)

requests_clean["is_escalated"] = (
    requests_clean["escalation_flag"].isin(["yes", "y", "true", "1"])
    | (requests_clean["service_status_clean"] == "Escalated")
)


# ------------------------------------------------------------
# 7. Identify open and closed service requests
# ------------------------------------------------------------

open_statuses = [
    "Open",
    "Pending",
    "In Progress",
    "Escalated"
]

closed_statuses = [
    "Closed",
    "Resolved"
]

requests_clean["is_open_request"] = (
    requests_clean["service_status_clean"].isin(open_statuses)
)

requests_clean["is_closed_request"] = (
    requests_clean["service_status_clean"].isin(closed_statuses)
)


# ------------------------------------------------------------
# 8. Identify SLA review cases
# ------------------------------------------------------------

# Assumption for this learning project:
# A request requiring more than 24 resolution hours
# is flagged for SLA review.
#
# This is a review threshold, not a claim about Jio's
# official operational SLA.

SLA_REVIEW_HOURS = 24

requests_clean["sla_review_flag"] = (
    requests_clean["resolution_time_hours"] > SLA_REVIEW_HOURS
)


# ------------------------------------------------------------
# 9. Identify invalid records
# ------------------------------------------------------------

requests_clean["invalid_request_date"] = (
    requests_clean["request_created_date"].isna()
)

requests_clean["invalid_resolution_time"] = (
    requests_clean["resolution_time_hours"].lt(0)
)

requests_clean["closed_before_created"] = (
    requests_clean["request_closed_date"].notna()
    & requests_clean["request_created_date"].notna()
    & (
        requests_clean["request_closed_date"]
        < requests_clean["request_created_date"]
    )
)


# ------------------------------------------------------------
# 10. Identify orphan complaint references
# ------------------------------------------------------------

valid_complaint_ids = set(
    complaints_clean["complaint_id"].dropna().unique()
)

requests_clean["orphan_complaint_reference"] = (
    ~requests_clean["complaint_id"].isin(valid_complaint_ids)
    | requests_clean["complaint_id"].isna()
)

orphan_requests = requests_clean.loc[
    requests_clean["orphan_complaint_reference"]
].copy()


# ------------------------------------------------------------
# 11. Create service request quality summary
# ------------------------------------------------------------

request_quality_summary = pd.DataFrame({
    "Metric": [
        "Original service request records",
        "Duplicate service request records removed",
        "Clean service request records",
        "Missing complaint IDs",
        "Orphan complaint references",
        "Invalid request creation dates",
        "Invalid resolution times",
        "Requests closed before creation",
        "Open service requests",
        "Closed service requests",
        "Escalated service requests",
        "SLA review cases"
    ],

    "Value": [
        len(requests_df),
        request_duplicate_count,
        len(requests_clean),
        requests_clean["complaint_id"].isna().sum(),
        requests_clean["orphan_complaint_reference"].sum(),
        requests_clean["invalid_request_date"].sum(),
        requests_clean["invalid_resolution_time"].sum(),
        requests_clean["closed_before_created"].sum(),
        requests_clean["is_open_request"].sum(),
        requests_clean["is_closed_request"].sum(),
        requests_clean["is_escalated"].sum(),
        requests_clean["sla_review_flag"].sum()
    ]
})


# ------------------------------------------------------------
# 12. Aggregate service requests by complaint ID
# ------------------------------------------------------------

# Keep all valid service request events in requests_clean.
# Exclude orphan complaint references from complaint-level
# aggregation because they cannot be matched to complaints.

requests_valid = requests_clean.loc[
    ~requests_clean["orphan_complaint_reference"]
].dropna(subset=["complaint_id"]).copy()

requests_by_complaint = (
    requests_valid
    .groupby("complaint_id")
    .agg(
        request_count=("service_request_id", "nunique"),

        open_request_count=("is_open_request", "sum"),

        closed_request_count=("is_closed_request", "sum"),

        escalated_request_count=("is_escalated", "sum"),

        sla_review_count=("sla_review_flag", "sum"),

        avg_resolution_time_hours=(
            "resolution_time_hours", "mean"
        ),

        max_resolution_time_hours=(
            "resolution_time_hours", "max"
        ),

        latest_request_date=("request_created_date", "max")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 13. Display service request quality summary
# ------------------------------------------------------------

print("\n========== SERVICE REQUEST QUALITY SUMMARY ==========")
display(request_quality_summary)


# ------------------------------------------------------------
# 14. Display status distribution
# ------------------------------------------------------------

print("\n========== SERVICE REQUEST STATUS DISTRIBUTION ==========")

display(
    requests_clean["service_status_clean"]
    .value_counts(dropna=False)
    .rename_axis("Service Status")
    .reset_index(name="Record Count")
)


# ------------------------------------------------------------
# 15. Display escalation and SLA analysis
# ------------------------------------------------------------

print("\n========== ESCALATION SUMMARY ==========")

display(
    requests_clean["is_escalated"]
    .value_counts()
    .rename_axis("Is Escalated")
    .reset_index(name="Record Count")
)

print("\n========== SLA REVIEW SUMMARY ==========")

display(
    requests_clean["sla_review_flag"]
    .value_counts()
    .rename_axis("SLA Review Required")
    .reset_index(name="Record Count")
)


# ------------------------------------------------------------
# 16. Display orphan complaint references
# ------------------------------------------------------------

print("\n========== ORPHAN SERVICE REQUESTS ==========")

print("Total Orphan Requests:", len(orphan_requests))

display(
    orphan_requests[
        [
            "service_request_id",
            "complaint_id",
            "customer_id",
            "service_status_clean"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 17. Display complaint-level aggregation
# ------------------------------------------------------------

print("\n========== REQUESTS BY COMPLAINT ==========")

display(requests_by_complaint.head(10))

print("\nRequests Clean Shape:", requests_clean.shape)

print(
    "Requests Aggregated Shape:",
    requests_by_complaint.shape
)

print(
    "Unique Complaint IDs:",
    requests_by_complaint["complaint_id"].nunique()
)


# ------------------------------------------------------------
# 18. Validation checks
# ------------------------------------------------------------

assert requests_clean["service_request_id"].is_unique, (
    "Duplicate service request IDs still exist."
)

assert requests_by_complaint["complaint_id"].is_unique, (
    "Duplicate complaint IDs found in request aggregation."
)

assert (
    requests_by_complaint["request_count"].sum()
    == requests_valid["service_request_id"].nunique()
), "Service request aggregation count does not reconcile."

print("\nMilestone 6 validation completed successfully!")

Original Service Request Shape: (1500, 10)
Duplicate Service Requests Removed: 0

========== SERVICE REQUEST QUALITY SUMMARY ==========


,Metric,Value
0,Original service request records,1500
1,Duplicate service request records removed,0
2,Clean service request records,1500
3,Missing complaint IDs,0
4,Orphan complaint references,3
5,Invalid request creation dates,0
6,Invalid resolution times,0
7,Requests closed before creation,0
8,Open service requests,857
9,Closed service requests,411



========== SERVICE REQUEST STATUS DISTRIBUTION ==========


,Service Status,Record Count
0,Unknown,232
1,Open,227
2,Escalated,218
3,Closed,214
4,Pending,207
5,In Progress,205
6,Resolved,197



========== ESCALATION SUMMARY ==========


...


========== ORPHAN SERVICE REQUESTS ==========
Total Orphan Requests: 3


,service_request_id,complaint_id,customer_id,service_status_clean
325,SR000326,CMPX943667,CUST00055,Pending
1138,SR001139,CMPX905803,CUST01012,Unknown
1492,SR001493,CMPX928838,CUST00696,Open



========== REQUESTS BY COMPLAINT ==========


,complaint_id,request_count,open_request_count,closed_request_count,escalated_request_count,sla_review_count,avg_resolution_time_hours,max_resolution_time_hours,latest_request_date
0,CMP000002,1,0,1,1,0,NaN,NaN,2026-04-30 18:00:00
1,CMP000003,1,0,1,0,1,102.0,102.0,2026-02-24 03:00:00
2,CMP000008,1,1,0,0,0,NaN,NaN,2026-05-19 17:00:00
3,CMP000009,2,1,0,1,0,19.0,19.0,2026-05-05 08:00:00
4,CMP000011,1,1,0,0,1,95.0,95.0,2026-03-30 17:00:00
5,CMP000013,2,1,1,1,0,4.0,4.0,2026-03-26 16:00:00
6,CMP000015,2,2,0,2,0,9.0,9.0,2026-05-24 19:00:00
7,CMP000032,1,1,0,1,0,NaN,NaN,2026-05-31 05:00:00
8,CMP000034,1,0,1,0,0,NaN,NaN,2026-04-25 04:00:00
9,CMP000035,1,0,1,1,0,NaN,NaN,2026-03-21 14:00:00



Requests Clean Shape: (1500, 19)
Requests Aggregated Shape: (1262, 9)
Unique Complaint IDs: 1262

Milestone 6 validation completed successfully!


## Milestone 07 · Analyze city, circle and repeated customer impact

**From the team:** Circle managers need geographic clusters and repeat contacts—not misleading counts created by multi-table joins.

**Your responsibility:** Use cleaned complaint facts and site context to summarize cities/circles and repeated customer complaints; preserve missing site references for review.

**Handover evidence:** Create `location_summary`, `repeat_customers` and `customer_repeat_count` at complaint grain.

**Reasoning:** City and circle summaries help locate concentrations of complaints, while repeat-customer counts identify repeated contact patterns. Site matches are checked separately so missing or unmatched inventory references remain visible instead of being mistaken for healthy network sites.

In [8]:
# Milestone 07 — Analyze city, circle and repeated customer impact
# Task 1: Use cleaned complaint facts and site context to summarize cities/circles and repeated customer complaints; preserve missing site references for review.
# Task 2: Produce this required evidence: Create `location_summary`, `repeat_customers` and `customer_repeat_count` at complaint grain.
# Implement your code below and display the requested outputs.
# ============================================================
# MILESTONE 7: LOCATION ANALYSIS & REPEAT CUSTOMER DETECTION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Create a location analysis dataset
# ------------------------------------------------------------

location_analysis = complaints_clean.copy()

print("Original Clean Complaints:", location_analysis.shape)


# ------------------------------------------------------------
# 2. Merge site inventory information
# ------------------------------------------------------------

# Select only the required site columns.
# sites_clean should contain one unique record per cell_site_id.

site_columns = [
    "cell_site_id",
    "site_name",
    "tower_type",
    "site_status",
    "avg_signal_strength_dbm",
    "capacity_utilization_pct"
]

site_context = sites_clean[site_columns].copy()

location_analysis = location_analysis.merge(
    site_context,
    on="cell_site_id",
    how="left",
    validate="many_to_one",
    indicator="_site_match"
)

# Preserve missing and unmatched site references.

location_analysis["missing_site_reference"] = (
    location_analysis["cell_site_id"].isna()
)

location_analysis["unmatched_site_reference"] = (
    location_analysis["_site_match"] == "left_only"
) & location_analysis["cell_site_id"].notna()

location_analysis["matched_site_reference"] = (
    location_analysis["_site_match"] == "both"
)

print("\nLocation Dataset Shape:", location_analysis.shape)


# ------------------------------------------------------------
# 3. Create site status indicators
# ------------------------------------------------------------

location_analysis["site_status_normalized"] = (
    location_analysis["site_status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

location_analysis["site_down_flag"] = (
    location_analysis["site_status_normalized"]
    .isin(["down", "inactive", "outage"])
)


# ------------------------------------------------------------
# 4. Create location summary
# ------------------------------------------------------------

location_summary = (
    location_analysis
    .groupby(
        ["city", "circle"],
        dropna=False
    )
    .agg(
        complaint_count=("complaint_id", "nunique"),

        unique_customers=("customer_id", "nunique"),

        unique_cell_sites=("cell_site_id", "nunique"),

        missing_site_reference_count=(
            "missing_site_reference", "sum"
        ),

        unmatched_site_reference_count=(
            "unmatched_site_reference", "sum"
        ),

        matched_site_complaint_count=(
            "matched_site_reference", "sum"
        ),

        site_down_complaint_count=(
            "site_down_flag", "sum"
        )
    )
    .reset_index()
)

# Sort locations by complaint volume.

location_summary = location_summary.sort_values(
    by="complaint_count",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 5. Calculate complaint count for each customer
# ------------------------------------------------------------

# Exclude missing customer IDs from customer-level analysis.

valid_customer_complaints = complaints_clean.dropna(
    subset=["customer_id"]
).copy()

customer_repeat_count = (
    valid_customer_complaints
    .groupby("customer_id")
    .agg(
        customer_complaint_count=("complaint_id", "nunique")
    )
    .reset_index()
)

customer_repeat_count = customer_repeat_count.sort_values(
    by="customer_complaint_count",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 6. Identify repeat customers
# ------------------------------------------------------------

# For exploratory analysis:
# A repeat customer has at least 2 distinct complaints.

repeat_customers = customer_repeat_count.loc[
    customer_repeat_count["customer_complaint_count"] >= 2
].copy()

repeat_customers = repeat_customers.sort_values(
    by="customer_complaint_count",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 7. Create customer repeat analysis summary
# ------------------------------------------------------------

repeat_customer_summary = pd.DataFrame({
    "Metric": [
        "Total unique customers",
        "Customers with one complaint",
        "Customers with multiple complaints",
        "Customers with 3 or more complaints",
        "Total complaints with valid customer IDs",
        "Complaints with missing customer IDs"
    ],

    "Value": [
        customer_repeat_count["customer_id"].nunique(),

        (
            customer_repeat_count["customer_complaint_count"] == 1
        ).sum(),

        (
            customer_repeat_count["customer_complaint_count"] >= 2
        ).sum(),

        (
            customer_repeat_count["customer_complaint_count"] >= 3
        ).sum(),

        valid_customer_complaints["complaint_id"].nunique(),

        complaints_clean["customer_id"].isna().sum()
    ]
})


# ------------------------------------------------------------
# 8. Display location summary
# ------------------------------------------------------------

print("\n========== LOCATION SUMMARY ==========")

display(location_summary.head(15))


# ------------------------------------------------------------
# 9. Display site reference analysis
# ------------------------------------------------------------

print("\n========== SITE REFERENCE ANALYSIS ==========")

site_reference_summary = pd.DataFrame({
    "Metric": [
        "Total complaints",
        "Missing site references",
        "Unmatched site references",
        "Matched site references",
        "Complaints associated with down/inactive/outage sites"
    ],

    "Value": [
        len(location_analysis),

        location_analysis["missing_site_reference"].sum(),

        location_analysis["unmatched_site_reference"].sum(),

        location_analysis["matched_site_reference"].sum(),

        location_analysis["site_down_flag"].sum()
    ]
})

display(site_reference_summary)


# ------------------------------------------------------------
# 10. Display repeat customer summary
# ------------------------------------------------------------

print("\n========== REPEAT CUSTOMER SUMMARY ==========")

display(repeat_customer_summary)


# ------------------------------------------------------------
# 11. Display top repeat customers
# ------------------------------------------------------------

print("\n========== TOP REPEAT CUSTOMERS ==========")

display(repeat_customers.head(15))


# ------------------------------------------------------------
# 12. Validation checks
# ------------------------------------------------------------

# One row per customer in customer_repeat_count.

assert customer_repeat_count["customer_id"].is_unique, (
    "Duplicate customer IDs found."
)

# Every valid customer complaint should be counted exactly once.

assert (
    customer_repeat_count["customer_complaint_count"].sum()
    == complaints_clean["customer_id"].notna().sum()
), "Customer complaint counts do not reconcile."

# Location merge must not multiply complaint records.

assert (
    len(location_analysis) == len(complaints_clean)
), "Location merge changed complaint row count."

assert (
    location_analysis["complaint_id"].is_unique
), "Duplicate complaint IDs found after location merge."

# Location summary complaint counts must reconcile.

assert (
    location_summary["complaint_count"].sum()
    == complaints_clean["complaint_id"].nunique()
), "Location complaint counts do not reconcile."

print("\nMilestone 7 validation completed successfully!")

Original Clean Complaints: (4901, 19)

Location Dataset Shape: (4901, 27)

========== LOCATION SUMMARY ==========


,city,circle,complaint_count,unique_customers,unique_cell_sites,missing_site_reference_count,unmatched_site_reference_count,matched_site_complaint_count,site_down_complaint_count
0,Indore,Madhya Pradesh,452,387,94,0,4,448,0
1,Pune,Maharashtra,447,379,91,0,2,445,0
2,Bengaluru,Karnataka,441,364,98,0,5,436,0
3,Delhi,Delhi NCR,440,365,95,0,4,436,0
4,Chennai,Tamil Nadu,430,356,92,0,1,429,0
5,Surat,Gujarat,428,368,85,0,1,427,0
6,Mumbai,Mumbai,418,345,84,0,1,417,0
7,Kolkata,West Bengal,418,360,83,0,3,415,0
8,Hyderabad,Andhra Pradesh & Telangana,381,326,79,0,1,380,0
9,Jaipur,Rajasthan,368,321,73,0,1,367,0



========== SITE REFERENCE ANALYSIS ==========


,Metric,Value
0,Total complaints,4901
1,Missing site references,0
2,Unmatched site references,29
3,Matched site references,4872
4,Complaints associated with down/inactive/outag...,0



========== REPEAT CUSTOMER SUMMARY ==========


,Metric,Value
0,Total unique customers,1217
1,Customers with one complaint,119
2,Customers with multiple complaints,1098
3,Customers with 3 or more complaints,926
4,Total complaints with valid customer IDs,4901
5,Complaints with missing customer IDs,0



========== TOP REPEAT CUSTOMERS ==========


,customer_id,customer_complaint_count
0,CUST01170,11
1,CUST00597,11
2,CUST00615,11
3,CUST00178,10
4,CUST01135,10
5,CUST00005,10
6,CUST01131,10
7,CUST00504,10
8,CUST00097,10
9,CUST00527,10



Milestone 7 validation completed successfully!


## Milestone 08 · Build the complaint-level investigation view

**From the team:** CareOps needs one dependable record per original complaint with network, customer, recharge and service evidence together.

**Your responsibility:** Merge customer and site dimensions plus complaint-aggregated evidence, preserve complaints without matches, fill only count metrics with zero, and reconcile row counts and IDs.

**Handover evidence:** Create `final_cases_df` with exactly one record for every distinct complaint.

**Reasoning:** The master dataset must preserve every cleaned complaint exactly once. Dimension tables are joined using validated many-to-one relationships, while event tables are pre-aggregated. Only count fields are filled with zero; unknown descriptive or monetary values are not fabricated.

In [9]:
# Milestone 08 — Build the complaint-level investigation view
# Task 1: Merge customer and site dimensions plus complaint-aggregated evidence, preserve complaints without matches, fill only count metrics with zero, and reconcile row counts and IDs.
# Task 2: Produce this required evidence: Create `final_cases_df` with exactly one record for every distinct complaint.
# Implement your code below and display the requested outputs.
# ============================================================
# MILESTONE 8: BUILD FINAL COMPLAINT-LEVEL INVESTIGATION VIEW
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Create the master dataset from clean complaints
# ------------------------------------------------------------

final_cases_df = complaints_clean.copy()

print("Original Clean Complaints:", len(final_cases_df))


# ------------------------------------------------------------
# 2. Validate primary complaint key
# ------------------------------------------------------------

assert final_cases_df["complaint_id"].notna().all()
assert final_cases_df["complaint_id"].is_unique

original_complaint_ids = set(
    final_cases_df["complaint_id"]
)


# ------------------------------------------------------------
# 3. Merge customer dimension
# ------------------------------------------------------------

customer_columns = [
    "customer_id",
    "customer_name",
    "customer_segment",
    "plan_type",
    "home_city",
    "circle",
    "account_age_days",
    "preferred_service",
    "monthly_recharge_value"
]

customer_dimension = customers_df[
    customer_columns
].copy()

# Rename overlapping fields to preserve complaint location.

customer_dimension = customer_dimension.rename(
    columns={
        "circle": "customer_circle",
        "home_city": "customer_home_city"
    }
)

assert customer_dimension["customer_id"].is_unique

final_cases_df = final_cases_df.merge(
    customer_dimension,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="_customer_match"
)

final_cases_df["customer_match_flag"] = (
    final_cases_df["_customer_match"] == "both"
)

final_cases_df = final_cases_df.drop(
    columns=["_customer_match"]
)

print("After Customer Merge:", final_cases_df.shape)


# ------------------------------------------------------------
# 4. Merge cell-site dimension
# ------------------------------------------------------------

site_columns = [
    "cell_site_id",
    "site_name",
    "tower_type",
    "site_status",
    "avg_signal_strength_dbm",
    "capacity_utilization_pct"
]

site_dimension = sites_clean[
    site_columns
].copy()

assert site_dimension["cell_site_id"].notna().all()
assert site_dimension["cell_site_id"].is_unique

final_cases_df = final_cases_df.merge(
    site_dimension,
    on="cell_site_id",
    how="left",
    validate="many_to_one",
    indicator="_site_match"
)

final_cases_df["site_match_flag"] = (
    final_cases_df["_site_match"] == "both"
)

final_cases_df = final_cases_df.drop(
    columns=["_site_match"]
)

print("After Site Merge:", final_cases_df.shape)


# ------------------------------------------------------------
# 5. Merge network evidence aggregated by cell site
# ------------------------------------------------------------

network_columns = [
    "cell_site_id",
    "network_report_count",
    "severe_network_report_count",
    "open_severe_network_report_count",
    "total_affected_customers",
    "avg_downtime_minutes"
]

network_evidence = network_by_site[
    network_columns
].copy()

assert network_evidence["cell_site_id"].notna().all()
assert network_evidence["cell_site_id"].is_unique

final_cases_df = final_cases_df.merge(
    network_evidence,
    on="cell_site_id",
    how="left",
    validate="many_to_one"
)

print("After Network Evidence Merge:", final_cases_df.shape)


# ------------------------------------------------------------
# 6. Merge recharge evidence by complaint
# ------------------------------------------------------------

recharge_columns = [
    "complaint_id",
    "recharge_count",
    "failed_recharge_count",
    "pending_recharge_count",
    "pending_refund_count",
    "failed_refund_count",
    "recharge_risk_count",
    "total_recharge_amount",
    "avg_recharge_amount",
    "latest_recharge_date"
]

recharge_evidence = recharge_by_complaint[
    recharge_columns
].copy()

assert recharge_evidence["complaint_id"].is_unique

final_cases_df = final_cases_df.merge(
    recharge_evidence,
    on="complaint_id",
    how="left",
    validate="one_to_one"
)

print("After Recharge Merge:", final_cases_df.shape)


# ------------------------------------------------------------
# 7. Merge service request evidence by complaint
# ------------------------------------------------------------

# Include all non-null complaint references in the aggregation.
# Orphan references will not match the master complaint table,
# but retaining them ensures source totals reconcile.

requests_for_merge = requests_clean.dropna(
    subset=["complaint_id"]
).copy()

requests_by_complaint = (
    requests_for_merge
    .groupby("complaint_id")
    .agg(
        request_count=("service_request_id", "nunique"),
        open_request_count=("is_open_request", "sum"),
        closed_request_count=("is_closed_request", "sum"),
        escalated_request_count=("is_escalated", "sum"),
        sla_review_count=("sla_review_flag", "sum"),
        avg_resolution_time_hours=(
            "resolution_time_hours", "mean"
        ),
        max_resolution_time_hours=(
            "resolution_time_hours", "max"
        ),
        latest_request_date=(
            "request_created_date", "max"
        )
    )
    .reset_index()
)

request_columns = [
    "complaint_id",
    "request_count",
    "open_request_count",
    "closed_request_count",
    "escalated_request_count",
    "sla_review_count",
    "avg_resolution_time_hours",
    "max_resolution_time_hours",
    "latest_request_date"
]

request_evidence = requests_by_complaint[
    request_columns
].copy()

assert request_evidence["complaint_id"].is_unique

final_cases_df = final_cases_df.merge(
    request_evidence,
    on="complaint_id",
    how="left",
    validate="one_to_one"
)

print("After Service Request Merge:", final_cases_df.shape)


# ------------------------------------------------------------
# 8. Merge customer repeat complaint counts
# ------------------------------------------------------------

repeat_evidence = customer_repeat_count[
    [
        "customer_id",
        "customer_complaint_count"
    ]
].copy()

assert repeat_evidence["customer_id"].is_unique

final_cases_df = final_cases_df.merge(
    repeat_evidence,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("After Repeat Customer Merge:", final_cases_df.shape)


# ------------------------------------------------------------
# 9. Fill missing values ONLY in count metrics
# ------------------------------------------------------------

count_columns = [
    "network_report_count",
    "severe_network_report_count",
    "open_severe_network_report_count",
    "total_affected_customers",

    "recharge_count",
    "failed_recharge_count",
    "pending_recharge_count",
    "pending_refund_count",
    "failed_refund_count",
    "recharge_risk_count",

    "request_count",
    "open_request_count",
    "closed_request_count",
    "escalated_request_count",
    "sla_review_count",

    "customer_complaint_count"
]

for col in count_columns:
    if col in final_cases_df.columns:
        final_cases_df[col] = (
            pd.to_numeric(
                final_cases_df[col],
                errors="coerce"
            )
            .fillna(0)
            .astype("int64")
        )


# ------------------------------------------------------------
# 10. Check final complaint-level dataset
# ------------------------------------------------------------

print("\n========== FINAL DATASET OVERVIEW ==========")

print("Final Dataset Shape:", final_cases_df.shape)

print(
    "Total Complaints:",
    final_cases_df["complaint_id"].nunique()
)

print(
    "Duplicate Complaint IDs:",
    final_cases_df["complaint_id"].duplicated().sum()
)

print(
    "Missing Customer IDs:",
    final_cases_df["customer_id"].isna().sum()
)

print(
    "Missing Cell Site IDs:",
    final_cases_df["cell_site_id"].isna().sum()
)


# ------------------------------------------------------------
# 11. Create merge quality summary
# ------------------------------------------------------------

merge_quality_summary = pd.DataFrame({
    "Metric": [
        "Original clean complaints",
        "Final master dataset records",
        "Unique complaint IDs",
        "Duplicate complaint IDs",
        "Customer dimension matches",
        "Unmatched customer records",
        "Site dimension matches",
        "Unmatched site records"
    ],

    "Value": [
        len(complaints_clean),
        len(final_cases_df),
        final_cases_df["complaint_id"].nunique(),
        final_cases_df["complaint_id"].duplicated().sum(),
        final_cases_df["customer_match_flag"].sum(),
        (~final_cases_df["customer_match_flag"]).sum(),
        final_cases_df["site_match_flag"].sum(),
        (~final_cases_df["site_match_flag"]).sum()
    ]
})

print("\n========== MERGE QUALITY SUMMARY ==========")

display(merge_quality_summary)


# ------------------------------------------------------------
# 12. Display selected master dataset columns
# ------------------------------------------------------------

preview_columns = [
    "complaint_id",
    "customer_id",
    "customer_name",
    "city",
    "circle",
    "issue_category",
    "complaint_status",
    "site_name",
    "site_status",
    "network_report_count",
    "severe_network_report_count",
    "recharge_count",
    "recharge_risk_count",
    "request_count",
    "escalated_request_count",
    "sla_review_count",
    "customer_complaint_count"
]

print("\n========== FINAL CASES PREVIEW ==========")

display(
    final_cases_df[
        preview_columns
    ].head(15)
)


# ------------------------------------------------------------
# 13. Final validation checks
# ------------------------------------------------------------

# Check 1: One row per complaint.

assert final_cases_df["complaint_id"].is_unique

# Check 2: No complaint IDs were lost or added.

assert (
    set(final_cases_df["complaint_id"])
    == original_complaint_ids
)

# Check 3: Final row count reconciles.

assert len(final_cases_df) == len(complaints_clean)

# Check 4: All required count columns are numeric.

for col in count_columns:
    assert pd.api.types.is_numeric_dtype(
        final_cases_df[col]
    )

# Check 5: Recharge counts reconcile for matched complaints.

recharge_expected = (
    recharge_clean.dropna(subset=["complaint_id"])
    .groupby("complaint_id")
    .size()
)

recharge_actual = final_cases_df.set_index(
    "complaint_id"
)["recharge_count"]

recharge_expected = recharge_expected.reindex(
    recharge_actual.index,
    fill_value=0
)

assert recharge_actual.eq(recharge_expected).all()

# Check 6: Service request counts reconcile for matched complaints.

request_expected = (
    requests_clean.dropna(subset=["complaint_id"])
    .groupby("complaint_id")
    .size()
)

request_actual = final_cases_df.set_index(
    "complaint_id"
)["request_count"]

request_expected = request_expected.reindex(
    request_actual.index,
    fill_value=0
)

assert request_actual.eq(request_expected).all()

print("\n============================================")
print("MILESTONE 8 VALIDATION COMPLETED SUCCESSFULLY!")
print("============================================")

print("\nFinal Master Dataset Shape:", final_cases_df.shape)
print("Unique Complaints:", final_cases_df["complaint_id"].nunique())
print("Duplicate Complaints:", final_cases_df["complaint_id"].duplicated().sum())

Original Clean Complaints: 4901
After Customer Merge: (4901, 28)
After Site Merge: (4901, 34)
After Network Evidence Merge: (4901, 39)
After Recharge Merge: (4901, 48)
After Service Request Merge: (4901, 56)
After Repeat Customer Merge: (4901, 57)

========== FINAL DATASET OVERVIEW ==========
Final Dataset Shape: (4901, 57)
Total Complaints: 4901
Duplicate Complaint IDs: 0
Missing Customer IDs: 0
Missing Cell Site IDs: 0

========== MERGE QUALITY SUMMARY ==========


,Metric,Value
0,Original clean complaints,4901
1,Final master dataset records,4901
2,Unique complaint IDs,4901
3,Duplicate complaint IDs,0
4,Customer dimension matches,4866
5,Unmatched customer records,35
6,Site dimension matches,4872
7,Unmatched site records,29



========== FINAL CASES PREVIEW ==========


,complaint_id,customer_id,customer_name,city,circle,issue_category,complaint_status,site_name,site_status,network_report_count,severe_network_report_count,recharge_count,recharge_risk_count,request_count,escalated_request_count,sla_review_count,customer_complaint_count
0,CMP000001,CUST00246,Ira Kulkarni,Jaipur,Rajasthan,Unknown,Open,JAI-NW-0878,Planned Upgrade,2,2,1,1,0,0,0,4
1,CMP000002,CUST00793,Ira Joshi,Ahmedabad,Gujarat,Unknown,Unknown,AHM-NW-0461,Under Maintenance,2,2,0,0,1,1,0,4
2,CMP000003,CUST00931,Meera Mehta,Hyderabad,Andhra Pradesh & Telangana,Unknown,In Progress,HYD-NW-0550,Congested,1,0,1,1,1,0,1,2
3,CMP000004,CUST00043,Ishaan Sharma,Ahmedabad,Gujarat,Internet Not Working,Open,AHM-NW-0443,Active,1,1,0,0,0,0,0,5
4,CMP000005,CUST00723,Reyansh Jain,Chennai,Tamil Nadu,Recharge Failure,Closed,CHE-NW-0214,Active,0,0,0,0,0,0,0,3
5,CMP000006,CUST00668,Kavya Nair,Lucknow,UP East,Signal Issue,Pending,LUC-NW-0682,Under Maintenance,1,1,0,0,0,0,0,6
6,CMP000007,CUST00745,Aarav Singh,Pune,Maharashtra,Service Request,Closed,PUN-NW-0922,Congested,1,1,1,1,0,0,0,6
7,CMP000008,CUST01112,Ananya Bose,Kolkata,West Bengal,Internet Not Working,Pending Review,KOL-NW-0668,Under Maintenance,1,1,0,0,1,0,0,4
8,CMP000009,CUST01060,Meera Reddy,Ahmedabad,Gujarat,Unresolved Complaint,Open,AHM-NW-0443,Active,1,1,1,1,2,1,0,4
9,CMP000010,CUST01178,Kavya Das,Hyderabad,Andhra Pradesh & Telangana,Slow Internet,Closed,HYD-NW-0801,Active,2,2,1,1,0,0,0,5



MILESTONE 8 VALIDATION COMPLETED SUCCESSFULLY!

Final Master Dataset Shape: (4901, 57)
Unique Complaints: 4901
Duplicate Complaints: 0


## Milestone 09 · Prioritize and recommend transparent next actions

**From the team:** Support leaders want explainable P0–P3 priorities based on four risk conditions, not an opaque model. They also need a draft message prompt without sending customer data externally.

**Your responsibility:** Assign P0 (3–4 signals), P1 (2), P2 (1), or P3 (0) using (1) open severe network incidents or down/inactive/outage site, (2) recharge risk, (3) escalated or SLA-review service requests, (4) three or more complaints by a customer. Produce case-specific next actions and offline AI prompt previews.

**Handover evidence:** Create four boolean risk signals, `case_priority`, `priority_score`, `recommended_action`, `ai_prompt_preview` in `final_cases_df`, and `priority_summary`.

**Reasoning:** A transparent score is easier for an operations team to audit than an opaque model. Each of the four Boolean signals contributes one point, and the score maps to P0–P3. The recommended action follows a documented precedence order. Prompt previews are generated locally; no customer data is sent to an external AI service.

In [10]:
# Milestone 09 — Prioritize and recommend transparent next actions
# Task 1: Create Boolean severe_network_signal: open severe network reports OR a site that is Down, Inactive, or Outage.
# Task 2: Create Boolean recharge_risk_signal from recharge_risk_count > 0.
# Task 3: Create Boolean service_followup_signal for escalated requests OR requests flagged for SLA review.
# Task 4: Create Boolean repeat_contact_signal for customers with at least three complaints.
# Task 5: Sum the four signals into priority_score and classify 3–4=P0, 2=P1, 1=P2, 0=P3.
# Task 6: Assign recommended_action by network, recharge, service and repeat-contact precedence.
# Task 7: Create privacy-safe ai_prompt_preview strings without calling an external API.
# Task 8: Create priority_summary with case_priority and case_count, and display priority breakdown and P0 cases.
# ============================================================
# MILESTONE 9: COMPLAINT PRIORITY SCORING & RECOMMENDED ACTIONS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Verify required columns
# ------------------------------------------------------------

required_columns = [
    "complaint_id",
    "site_status",
    "severe_network_report_count",
    "recharge_risk_count",
    "escalated_request_count",
    "sla_review_count",
    "customer_complaint_count"
]

missing_columns = [
    col for col in required_columns
    if col not in final_cases_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

print("Required columns verified successfully.")


# ------------------------------------------------------------
# 2. Create severe network risk signal
# ------------------------------------------------------------

# A complaint receives network risk when:
# 1. A severe network report exists, OR
# 2. The associated site is Down, Inactive or Outage.

site_status_normalized = (
    final_cases_df["site_status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

final_cases_df["severe_network_signal"] = (
    pd.to_numeric(
        final_cases_df["severe_network_report_count"],
        errors="coerce"
    ).fillna(0).gt(0)
    |
    site_status_normalized.isin(
        ["down", "inactive", "outage"]
    )
).fillna(False).astype(bool)


# ------------------------------------------------------------
# 3. Create recharge risk signal
# ------------------------------------------------------------

final_cases_df["recharge_risk_signal"] = (
    pd.to_numeric(
        final_cases_df["recharge_risk_count"],
        errors="coerce"
    ).fillna(0).gt(0)
).astype(bool)


# ------------------------------------------------------------
# 4. Create service follow-up signal
# ------------------------------------------------------------

final_cases_df["service_followup_signal"] = (
    pd.to_numeric(
        final_cases_df["escalated_request_count"],
        errors="coerce"
    ).fillna(0).gt(0)
    |
    pd.to_numeric(
        final_cases_df["sla_review_count"],
        errors="coerce"
    ).fillna(0).gt(0)
).astype(bool)


# ------------------------------------------------------------
# 5. Create repeat contact signal
# ------------------------------------------------------------

final_cases_df["repeat_contact_signal"] = (
    pd.to_numeric(
        final_cases_df["customer_complaint_count"],
        errors="coerce"
    ).fillna(0).ge(3)
).astype(bool)


# ------------------------------------------------------------
# 6. Calculate priority score
# ------------------------------------------------------------

risk_signal_columns = [
    "severe_network_signal",
    "recharge_risk_signal",
    "service_followup_signal",
    "repeat_contact_signal"
]

# Each True signal contributes 1 point.

final_cases_df["priority_score"] = (
    final_cases_df[risk_signal_columns]
    .sum(axis=1)
    .astype(int)
)


# ------------------------------------------------------------
# 7. Assign complaint priority
# ------------------------------------------------------------

def assign_priority(score):

    if score >= 3:
        return "P0"

    elif score == 2:
        return "P1"

    elif score == 1:
        return "P2"

    else:
        return "P3"


final_cases_df["case_priority"] = (
    final_cases_df["priority_score"]
    .apply(assign_priority)
)


# ------------------------------------------------------------
# 8. Assign recommended actions
# ------------------------------------------------------------

# Precedence:
# Network > Recharge > Service > Repeat Contact

def recommend_action(row):

    if row["severe_network_signal"]:
        return (
            "Escalate to Network Operations. "
            "Investigate the affected cell site and "
            "verify network restoration."
        )

    elif row["recharge_risk_signal"]:
        return (
            "Assign to Recharge Support. "
            "Verify the failed transaction and "
            "investigate pending or failed refunds."
        )

    elif row["service_followup_signal"]:
        return (
            "Escalate to Customer Service Operations. "
            "Review the pending service request and "
            "complete the required follow-up."
        )

    elif row["repeat_contact_signal"]:
        return (
            "Assign to Senior Customer Support. "
            "Review the customer's complaint history "
            "and provide a consolidated resolution."
        )

    else:
        return (
            "Continue standard complaint handling. "
            "Monitor the case and update its resolution status."
        )


final_cases_df["recommended_action"] = (
    final_cases_df.apply(
        recommend_action,
        axis=1
    )
)


# ------------------------------------------------------------
# 9. Create privacy-safe AI prompt previews
# ------------------------------------------------------------

# These are offline draft prompts only.
# No customer personal information is included.
# No external AI API is called.

def create_ai_prompt(row):

    return (
        "You are a telecom customer operations analyst.\n"
        "Review the following anonymized complaint evidence "
        "and suggest a concise support action.\n\n"

        f"Issue Category: {row['issue_category']}\n"
        f"Issue Group: {row['issue_group']}\n"
        f"City: {row['city']}\n"
        f"Circle: {row['circle']}\n"
        f"Complaint Status: {row['complaint_status']}\n"
        f"Priority: {row['case_priority']}\n"
        f"Priority Score: {row['priority_score']}\n\n"

        f"Network Risk: {row['severe_network_signal']}\n"
        f"Recharge Risk: {row['recharge_risk_signal']}\n"
        f"Service Follow-up: {row['service_followup_signal']}\n"
        f"Repeat Contact: {row['repeat_contact_signal']}\n\n"

        f"Network Reports: {row['network_report_count']}\n"
        f"Recharge Risk Count: {row['recharge_risk_count']}\n"
        f"Escalated Requests: {row['escalated_request_count']}\n"
        f"SLA Review Count: {row['sla_review_count']}\n\n"

        f"Recommended Action: {row['recommended_action']}\n\n"

        "Provide a short investigation summary and "
        "the next recommended operational step. "
        "Do not infer missing information or request "
        "personal customer data."
    )


final_cases_df["ai_prompt_preview"] = (
    final_cases_df.apply(
        create_ai_prompt,
        axis=1
    )
)


# ------------------------------------------------------------
# 10. Create priority summary
# ------------------------------------------------------------

priority_order = ["P0", "P1", "P2", "P3"]

priority_summary = (
    final_cases_df["case_priority"]
    .value_counts()
    .reindex(priority_order, fill_value=0)
    .rename_axis("case_priority")
    .reset_index(name="case_count")
)


# ------------------------------------------------------------
# 11. Display priority distribution
# ------------------------------------------------------------

print("\n========== COMPLAINT PRIORITY SUMMARY ==========")

display(priority_summary)


# ------------------------------------------------------------
# 12. Display risk signal distribution
# ------------------------------------------------------------

print("\n========== RISK SIGNAL DISTRIBUTION ==========")

risk_summary = pd.DataFrame({
    "Risk Signal": [
        "Severe Network Risk",
        "Recharge Risk",
        "Service Follow-up",
        "Repeat Contact"
    ],

    "Affected Complaints": [
        final_cases_df["severe_network_signal"].sum(),
        final_cases_df["recharge_risk_signal"].sum(),
        final_cases_df["service_followup_signal"].sum(),
        final_cases_df["repeat_contact_signal"].sum()
    ]
})

display(risk_summary)


# ------------------------------------------------------------
# 13. Display P0 critical complaints
# ------------------------------------------------------------

print("\n========== P0 CRITICAL COMPLAINTS ==========")

p0_cases = final_cases_df.loc[
    final_cases_df["case_priority"] == "P0"
].copy()

p0_columns = [
    "complaint_id",
    "city",
    "circle",
    "issue_category",
    "complaint_status",
    "priority_score",
    "severe_network_signal",
    "recharge_risk_signal",
    "service_followup_signal",
    "repeat_contact_signal",
    "recommended_action"
]

display(
    p0_cases[p0_columns].head(20)
)

print("Total P0 Complaints:", len(p0_cases))


# ------------------------------------------------------------
# 14. Display recommended action distribution
# ------------------------------------------------------------

print("\n========== RECOMMENDED ACTION DISTRIBUTION ==========")

action_summary = (
    final_cases_df["recommended_action"]
    .value_counts()
    .rename_axis("Recommended Action")
    .reset_index(name="Complaint Count")
)

display(action_summary)


# ------------------------------------------------------------
# 15. Display AI prompt preview
# ------------------------------------------------------------

print("\n========== AI PROMPT PREVIEW ==========")

if len(final_cases_df) > 0:

    print(
        final_cases_df["ai_prompt_preview"].iloc[0]
    )


# ------------------------------------------------------------
# 16. Create priority-wise operational summary
# ------------------------------------------------------------

priority_operations_summary = (
    final_cases_df
    .groupby(
        ["case_priority", "issue_group"],
        dropna=False
    )
    .agg(
        case_count=("complaint_id", "nunique"),

        avg_priority_score=("priority_score", "mean"),

        unique_customers=("customer_id", "nunique")
    )
    .reset_index()
)

priority_operations_summary["case_priority"] = pd.Categorical(
    priority_operations_summary["case_priority"],
    categories=priority_order,
    ordered=True
)

priority_operations_summary = (
    priority_operations_summary
    .sort_values(
        ["case_priority", "case_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("\n========== PRIORITY BY ISSUE GROUP ==========")

display(priority_operations_summary)


# ------------------------------------------------------------
# 17. Validation checks
# ------------------------------------------------------------

# Check 1: All four signals must be Boolean.

for col in risk_signal_columns:

    assert pd.api.types.is_bool_dtype(
        final_cases_df[col]
    ), f"{col} is not Boolean."


# Check 2: Priority score must equal sum of signals.

expected_score = (
    final_cases_df[risk_signal_columns]
    .sum(axis=1)
)

assert final_cases_df["priority_score"].eq(
    expected_score
).all()


# Check 3: Priority classification must be correct.

expected_priority = expected_score.map(
    lambda x: (
        "P0" if x >= 3 else
        "P1" if x == 2 else
        "P2" if x == 1 else
        "P3"
    )
)

assert final_cases_df["case_priority"].eq(
    expected_priority
).all()


# Check 4: Priority summary must reconcile.

assert (
    priority_summary["case_count"].sum()
    == len(final_cases_df)
)


# Check 5: Complaint IDs must remain unique.

assert final_cases_df["complaint_id"].is_unique


# Check 6: AI prompt preview must exist.

assert final_cases_df["ai_prompt_preview"].notna().all()


# Check 7: Recommended actions must exist.

assert final_cases_df["recommended_action"].notna().all()


print("\n==============================================")
print("MILESTONE 9 VALIDATION COMPLETED SUCCESSFULLY!")
print("==============================================")

print("Total Complaints:", len(final_cases_df))
print("Total P0 Cases:", len(p0_cases))
print("Total P1 Cases:", (final_cases_df["case_priority"] == "P1").sum())
print("Total P2 Cases:", (final_cases_df["case_priority"] == "P2").sum())
print("Total P3 Cases:", (final_cases_df["case_priority"] == "P3").sum())

Required columns verified successfully.

========== COMPLAINT PRIORITY SUMMARY ==========


,case_priority,case_count
0,P0,990
1,P1,2370
2,P2,1417
3,P3,124



========== RISK SIGNAL DISTRIBUTION ==========


,Risk Signal,Affected Complaints
0,Severe Network Risk,2880
1,Recharge Risk,1024
2,Service Follow-up,898
3,Repeat Contact,4438



========== P0 CRITICAL COMPLAINTS ==========


...

Total P0 Complaints: 990

========== RECOMMENDED ACTION DISTRIBUTION ==========


,Recommended Action,Complaint Count
0,Escalate to Network Operations. Investigate th...,2880
1,Assign to Senior Customer Support. Review the ...,1187
2,Assign to Recharge Support. Verify the failed ...,413
3,Escalate to Customer Service Operations. Revie...,297
4,Continue standard complaint handling. Monitor ...,124



========== AI PROMPT PREVIEW ==========
You are a telecom customer operations analyst.
Review the following anonymized complaint evidence and suggest a concise support action.

Issue Category: Unknown
Issue Group: Unknown
City: Jaipur
Circle: Rajasthan
Complaint Status: Open
Priority: P0
Priority Score: 3

Network Risk: True
Recharge Risk: True
Service Follow-up: False
Repeat Contact: True

Network Reports: 2
Recharge Risk Count: 1
Escalated Requests: 0
SLA Review Count: 0

Recommended Action: Escalate to Network Operations. Investigate the affected cell site and verify network restoration.

Provide a short investigation summary and the next recommended operational step. Do not infer missing information or request personal customer data.

========== PRIORITY BY ISSUE GROUP ==========


,case_priority,issue_group,case_count,avg_priority_score,unique_customers
0,P0,Network,356,3.115169,304
1,P0,Recharge,225,3.120000,202
2,P0,Data,168,3.107143,156
3,P0,Unknown,99,3.101010,91
4,P0,Service Request,90,3.111111,86
5,P0,General,28,3.107143,28
6,P0,Billing,24,3.166667,23
7,P1,Network,857,2.000000,603
8,P1,Recharge,492,2.000000,402
9,P1,Data,415,2.000000,342



MILESTONE 9 VALIDATION COMPLETED SUCCESSFULLY!
Total Complaints: 4901
Total P0 Cases: 990
Total P1 Cases: 2370
Total P2 Cases: 1417
Total P3 Cases: 124


## Milestone 10 · Deliver operational reports and a searchable case view

**From the team:** The CareOps supervisor needs an exportable complaint queue, circle summary and quick filtering tool that works inside Colab without external APIs.

**Your responsibility:** Sort in P0, P1, P2, P3 order. Export `jio_complaint_priority_report.csv` and `jio_careops_operations_summary.xlsx`, offer an optional widget-based search/filter with console fallback, and write 3–5 data-backed findings.

**Handover evidence:** Create `final_report`, `circle_summary`, `priority_summary`, CSV/XLSX exports and `filter_cases` helper.

**Reasoning:** The final report turns analysis into an actionable queue. Sorting by priority makes urgent cases visible, grouped summaries support circle-level review, and reusable filters help investigate subsets without changing the underlying report. CSV and Excel exports make the outputs portable.

In [11]:
# Milestone 10 — Deliver operational reports and a searchable case view
# Task 1: Create final_report with one row per complaint and sort by priority P0, P1, P2, P3.
# Task 2: Build circle_summary by circle and case_priority with case_count and unique_customers.
# Task 3: Export final_report to DATA_DIR/'jio_complaint_priority_report.csv'.
# Task 4: Export priority_summary and circle_summary as Excel worksheets in DATA_DIR/'jio_careops_operations_summary.xlsx'.
# Task 5: Provide callable filter_cases to search/filter the report by priority, circle, city, issue group and IDs.
# Task 6: Display saved files and report previews, and write 3–5 evidence-based handover findings.
# ============================================================
# MILESTONE 10: FINAL REPORTING, EXPORT & BUSINESS INSIGHTS
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. Validate final dataset
# ------------------------------------------------------------

assert "final_cases_df" in globals(), (
    "final_cases_df not found. Complete Milestone 8 first."
)

required_columns = [
    "complaint_id",
    "customer_id",
    "city",
    "circle",
    "issue_group",
    "case_priority",
    "priority_score",
    "recommended_action"
]

missing_columns = [
    col for col in required_columns
    if col not in final_cases_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

assert final_cases_df["complaint_id"].is_unique

print("Final dataset verified:", final_cases_df.shape)


# ------------------------------------------------------------
# 2. Create final priority report
# ------------------------------------------------------------

priority_order = ["P0", "P1", "P2", "P3"]

final_report = final_cases_df.copy()

# Assign a numeric sorting order.

priority_rank = {
    "P0": 0,
    "P1": 1,
    "P2": 2,
    "P3": 3
}

final_report["priority_rank"] = (
    final_report["case_priority"]
    .map(priority_rank)
)

# Sort P0 first, then higher scores first.

sort_columns = [
    "priority_rank",
    "priority_score"
]

ascending_order = [True, False]

if "complaint_date" in final_report.columns:
    sort_columns.append("complaint_date")
    ascending_order.append(True)

final_report = (
    final_report
    .sort_values(
        by=sort_columns,
        ascending=ascending_order,
        na_position="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Create circle-wise priority summary
# ------------------------------------------------------------

circle_summary = (
    final_report
    .groupby(
        ["circle", "case_priority"],
        dropna=False
    )
    .agg(
        case_count=("complaint_id", "nunique"),
        unique_customers=("customer_id", "nunique")
    )
    .reset_index()
)

circle_summary["case_priority"] = pd.Categorical(
    circle_summary["case_priority"],
    categories=priority_order,
    ordered=True
)

circle_summary = (
    circle_summary
    .sort_values(
        ["circle", "case_priority"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Create overall priority summary
# ------------------------------------------------------------

priority_summary = (
    final_report["case_priority"]
    .value_counts()
    .reindex(priority_order, fill_value=0)
    .rename_axis("case_priority")
    .reset_index(name="case_count")
)

priority_summary["percentage"] = (
    priority_summary["case_count"]
    / len(final_report) * 100
).round(2)


# ------------------------------------------------------------
# 5. Create risk signal summary
# ------------------------------------------------------------

risk_columns = {
    "Severe Network Risk": "severe_network_signal",
    "Recharge Risk": "recharge_risk_signal",
    "Service Follow-up": "service_followup_signal",
    "Repeat Contact": "repeat_contact_signal"
}

risk_summary = pd.DataFrame({
    "Risk Signal": list(risk_columns.keys()),

    "Affected Complaints": [
        final_report[col].fillna(False).astype(bool).sum()
        for col in risk_columns.values()
    ]
})


# ------------------------------------------------------------
# 6. Create recommended action summary
# ------------------------------------------------------------

action_summary = (
    final_report["recommended_action"]
    .value_counts()
    .rename_axis("recommended_action")
    .reset_index(name="complaint_count")
)


# ------------------------------------------------------------
# 7. Create reusable complaint filtering function
# ------------------------------------------------------------

def filter_cases(
    priority=None,
    circle=None,
    city=None,
    issue_group=None,
    complaint_ids=None
):
    """
    Filter complaint records using optional parameters.

    Parameters:
        priority      : P0, P1, P2, P3
        circle        : Circle name or list of names
        city          : City name or list of names
        issue_group   : Issue group or list of groups
        complaint_ids : Complaint ID or list of IDs

    Returns:
        Filtered pandas DataFrame.
    """

    result = final_report.copy()

    def convert_to_list(value):
        if value is None:
            return None

        if isinstance(value, (list, tuple, set)):
            return list(value)

        return [value]

    # Priority filter

    if priority is not None:
        values = convert_to_list(priority)

        values = [
            str(v).strip().upper()
            for v in values
        ]

        result = result[
            result["case_priority"].isin(values)
        ]

    # Circle filter

    if circle is not None:
        values = convert_to_list(circle)

        values = [
            str(v).strip().casefold()
            for v in values
        ]

        result = result[
            result["circle"]
            .astype("string")
            .str.strip()
            .str.casefold()
            .isin(values)
        ]

    # City filter

    if city is not None:
        values = convert_to_list(city)

        values = [
            str(v).strip().casefold()
            for v in values
        ]

        result = result[
            result["city"]
            .astype("string")
            .str.strip()
            .str.casefold()
            .isin(values)
        ]

    # Issue group filter

    if issue_group is not None:
        values = convert_to_list(issue_group)

        values = [
            str(v).strip().casefold()
            for v in values
        ]

        result = result[
            result["issue_group"]
            .astype("string")
            .str.strip()
            .str.casefold()
            .isin(values)
        ]

    # Complaint ID filter

    if complaint_ids is not None:
        values = convert_to_list(complaint_ids)

        values = [
            str(v).strip().casefold()
            for v in values
        ]

        result = result[
            result["complaint_id"]
            .astype("string")
            .str.strip()
            .str.casefold()
            .isin(values)
        ]

    return result.reset_index(drop=True)


# ------------------------------------------------------------
# 8. Demonstrate complaint filtering
# ------------------------------------------------------------

print("\n========== FILTER FUNCTION TESTS ==========")

print("\nTest 1: P0 Complaints")

p0_filtered = filter_cases(priority="P0")

print("Records Found:", len(p0_filtered))

display(
    p0_filtered[
        [
            "complaint_id",
            "city",
            "circle",
            "issue_category",
            "case_priority",
            "priority_score"
        ]
    ].head(10)
)


print("\nTest 2: Complaints in West Bengal Circle")

west_bengal_cases = filter_cases(
    circle="West Bengal"
)

print("Records Found:", len(west_bengal_cases))

display(west_bengal_cases.head(10))


print("\nTest 3: Network Issue Group")

network_cases = filter_cases(
    issue_group="Network"
)

print("Records Found:", len(network_cases))

display(network_cases.head(10))


print("\nTest 4: P0 Complaints in Kolkata")

kolkata_p0_cases = filter_cases(
    priority="P0",
    city="Kolkata"
)

print("Records Found:", len(kolkata_p0_cases))

display(kolkata_p0_cases.head(10))


# ------------------------------------------------------------
# 9. Generate business insights
# ------------------------------------------------------------

total_complaints = len(final_report)

p0_count = (
    final_report["case_priority"] == "P0"
).sum()

p1_count = (
    final_report["case_priority"] == "P1"
).sum()

critical_high_count = p0_count + p1_count

critical_high_percentage = (
    critical_high_count / total_complaints * 100
    if total_complaints > 0 else 0
)

# Top circle

circle_volume = (
    final_report
    .groupby("circle", dropna=False)
    ["complaint_id"]
    .nunique()
    .sort_values(ascending=False)
)

# Top city

city_volume = (
    final_report
    .groupby("city", dropna=False)
    ["complaint_id"]
    .nunique()
    .sort_values(ascending=False)
)

# Top issue group

issue_volume = (
    final_report
    .groupby("issue_group", dropna=False)
    ["complaint_id"]
    .nunique()
    .sort_values(ascending=False)
)

# Top risk signal

top_risk = risk_summary.sort_values(
    "Affected Complaints",
    ascending=False
).iloc[0]

# Top recommended action

top_action = action_summary.iloc[0]


business_findings = [
    {
        "Finding": "Overall complaint volume",
        "Observation": (
            f"The dataset contains {total_complaints:,} "
            "unique complaints."
        )
    },
    {
        "Finding": "Critical and high-priority workload",
        "Observation": (
            f"{critical_high_count:,} complaints "
            f"({critical_high_percentage:.2f}%) "
            "are classified as P0 or P1."
        )
    },
    {
        "Finding": "Highest complaint circle",
        "Observation": (
            f"{circle_volume.index[0]} has the highest "
            f"complaint volume with "
            f"{circle_volume.iloc[0]:,} complaints."
        ) if len(circle_volume) else "No circle data available."
    },
    {
        "Finding": "Most frequent risk signal",
        "Observation": (
            f"{top_risk['Risk Signal']} affects "
            f"{int(top_risk['Affected Complaints']):,} complaints."
        )
    },
    {
        "Finding": "Highest complaint city and issue group",
        "Observation": (
            f"{city_volume.index[0]} is the highest-volume city "
            f"({city_volume.iloc[0]:,} complaints). "
            f"{issue_volume.index[0]} is the largest issue group "
            f"({issue_volume.iloc[0]:,} complaints)."
        ) if len(city_volume) and len(issue_volume)
        else "City or issue-group data unavailable."
    }
]

business_findings_df = pd.DataFrame(
    business_findings
)

print("\n========== BUSINESS FINDINGS ==========")

display(business_findings_df)


# ------------------------------------------------------------
# 10. Prepare final report columns
# ------------------------------------------------------------

# Keep all columns in final_report.
# The CSV will contain the full complaint-level investigation data.

print("\n========== FINAL REPORT PREVIEW ==========")

display(final_report.head(15))

print("\nFinal Report Shape:", final_report.shape)


# ------------------------------------------------------------
# 11. Export CSV report
# ------------------------------------------------------------

csv_filename = "jio_complaint_priority_report.csv"

final_report.to_csv(
    csv_filename,
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV report exported:", csv_filename)


# ------------------------------------------------------------
# 12. Export Excel operations summary
# ------------------------------------------------------------

excel_filename = "jio_careops_operations_summary.xlsx"

with pd.ExcelWriter(
    excel_filename,
    engine="openpyxl"
) as writer:

    final_report.to_excel(
        writer,
        sheet_name="Priority Report",
        index=False
    )

    priority_summary.to_excel(
        writer,
        sheet_name="Priority Summary",
        index=False
    )

    circle_summary.to_excel(
        writer,
        sheet_name="Circle Summary",
        index=False
    )

    risk_summary.to_excel(
        writer,
        sheet_name="Risk Summary",
        index=False
    )

    action_summary.to_excel(
        writer,
        sheet_name="Action Summary",
        index=False
    )

    business_findings_df.to_excel(
        writer,
        sheet_name="Business Findings",
        index=False
    )

print("Excel operations summary exported:", excel_filename)


# ------------------------------------------------------------
# 13. Display exported file details
# ------------------------------------------------------------

exported_files = pd.DataFrame({
    "File Name": [
        csv_filename,
        excel_filename
    ],

    "File Size (KB)": [
        round(os.path.getsize(csv_filename) / 1024, 2),
        round(os.path.getsize(excel_filename) / 1024, 2)
    ],

    "Export Status": [
        "Successful" if os.path.exists(csv_filename) else "Failed",
        "Successful" if os.path.exists(excel_filename) else "Failed"
    ]
})

print("\n========== EXPORTED FILES ==========")

display(exported_files)


# ------------------------------------------------------------
# 14. Final validation
# ------------------------------------------------------------

# Check priority report

assert len(final_report) == len(final_cases_df)

assert final_report["complaint_id"].is_unique

assert final_report["case_priority"].isin(
    priority_order
).all()

# Check priority counts

assert (
    priority_summary["case_count"].sum()
    == len(final_report)
)

# Check circle summary

assert (
    circle_summary["case_count"].sum()
    == len(final_report)
)

# Check filtering function

assert len(filter_cases()) == len(final_report)

assert (
    len(filter_cases(priority="P0"))
    == (final_report["case_priority"] == "P0").sum()
)

# Check exported files

assert os.path.exists(csv_filename)

assert os.path.exists(excel_filename)

assert os.path.getsize(csv_filename) > 0

assert os.path.getsize(excel_filename) > 0

print("\n==============================================")
print("MILESTONE 10 VALIDATION COMPLETED SUCCESSFULLY!")
print("==============================================")

print("Total Complaints:", len(final_report))

print("CSV Export:", csv_filename)

print("Excel Export:", excel_filename)

print("Total Business Findings:", len(business_findings_df))

print("All final validation checks passed!")

Final dataset verified: (4901, 65)

========== FILTER FUNCTION TESTS ==========

Test 1: P0 Complaints
Records Found: 990


,complaint_id,city,circle,issue_category,case_priority,priority_score
0,CMP004198,Bengaluru,Karnataka,Signal Issue,P0,4
1,CMP000464,Delhi,Delhi NCR,Signal Issue,P0,4
2,CMP002955,Chennai,Tamil Nadu,Service Request,P0,4
3,CMP000655,Jaipur,Rajasthan,Network Down,P0,4
4,CMP002289,Jaipur,Rajasthan,Service Request,P0,4
5,CMP004903,Hyderabad,Andhra Pradesh & Telangana,Service Request,P0,4
6,CMP004521,Delhi,Delhi NCR,Network Down,P0,4
7,CMP004719,Hyderabad,Andhra Pradesh & Telangana,Internet Not Working,P0,4
8,CMP004845,Jaipur,Rajasthan,Signal Issue,P0,4
9,CMP004672,Chennai,Tamil Nadu,Slow Internet,P0,4



Test 2: Complaints in West Bengal Circle
Records Found: 418


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,...,customer_complaint_count,severe_network_signal,recharge_risk_signal,service_followup_signal,repeat_contact_signal,priority_score,case_priority,recommended_action,ai_prompt_preview,priority_rank
0,CMP003326,CUST00154,SITE0030,2026-02-19,02:23,Kolkata,West Bengal,WhatsApp Bot,Recharge Failure,Repeated complaint from same location and serv...,...,5,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
1,CMP001502,CUST00264,SITE0826,2026-02-24,11:04,Kolkata,West Bengal,Call Center,Signal Issue,Repeated complaint from same location and serv...,...,4,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
2,CMP002525,CUST00044,SITE0401,2026-03-05,01:42,Kolkata,West Bengal,Call Center,Network Down,Fiber connection outage reported for apartment...,...,9,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
3,CMP002958,CUST00647,SITE0993,2026-03-07,05:40,Kolkata,West Bengal,Retail Store,Service Request,SIM activation or verification is pending afte...,...,5,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
4,CMP000015,CUST00301,SITE0932,2026-04-06,12:30,Kolkata,West Bengal,Social Media,Signal Issue,Fiber connection outage reported for apartment...,...,4,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
5,CMP004112,CUST00238,SITE0518,2026-04-07,19:23,Kolkata,West Bengal,Social Media,Billing Issue,Fiber connection outage reported for apartment...,...,6,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
6,CMP000481,CUST00920,SITE0635,2026-04-20,05:13,Kolkata,West Bengal,Email,Recharge Failure,Payment was debited but recharge benefit is no...,...,5,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
7,CMP000449,CUST00593,SITE0951,2026-04-22,18:32,Kolkata,West Bengal,MyJio App,Unknown,Customer says calls drop during office hours.,...,4,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
8,CMP003537,CUST00010,SITE0756,2026-04-28,04:18,Kolkata,West Bengal,Email,Slow Internet,Customer reports unstable network for the last...,...,7,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
9,CMP000513,CUST00122,SITE0259,2026-05-21,20:13,Kolkata,West Bengal,WhatsApp Bot,Internet Not Working,Payment was debited but recharge benefit is no...,...,5,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0



Test 3: Network Issue Group
Records Found: 1719


...


========== FINAL REPORT PREVIEW ==========


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,...,customer_complaint_count,severe_network_signal,recharge_risk_signal,service_followup_signal,repeat_contact_signal,priority_score,case_priority,recommended_action,ai_prompt_preview,priority_rank
0,CMP004198,CUST00508,SITE0838,2026-02-03,08:42,Bengaluru,Karnataka,Call Center,Signal Issue,Repeated complaint from same location and serv...,...,7,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
1,CMP000464,CUST00751,SITE0273,2026-02-04,21:18,Delhi,Delhi NCR,Retail Store,Signal Issue,Fiber connection outage reported for apartment...,...,5,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
2,CMP002955,CUST00367,SITE0405,2026-02-05,10:12,Chennai,Tamil Nadu,MyJio App,Service Request,Customer reports unstable network for the last...,...,3,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
3,CMP000655,CUST01133,SITE0617,2026-02-06,08:34,Jaipur,Rajasthan,Retail Store,Network Down,Customer is unable to use data indoors.,...,6,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
4,CMP002289,CUST00656,SITE0409,2026-02-06,22:00,Jaipur,Rajasthan,WhatsApp Bot,Service Request,Customer reports unstable network for the last...,...,5,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
5,CMP004903,CUST00074,SITE0992,2026-02-06,06:20,Hyderabad,Andhra Pradesh & Telangana,Retail Store,Service Request,Customer is unable to use data indoors.,...,6,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
6,CMP004521,CUST00601,SITE0289,2026-02-07,14:50,Delhi,Delhi NCR,Social Media,Network Down,Customer says calls drop during office hours.,...,9,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
7,CMP004719,CUST01128,SITE0199,2026-02-07,22:17,Hyderabad,Andhra Pradesh & Telangana,Social Media,Internet Not Working,Customer reports unstable network for the last...,...,6,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
8,CMP004845,CUST00233,SITE0492,2026-02-10,05:52,Jaipur,Rajasthan,WhatsApp Bot,Signal Issue,Payment was debited but recharge benefit is no...,...,4,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
9,CMP004672,CUST01195,SITE0628,2026-02-11,19:36,Chennai,Tamil Nadu,Email,Slow Internet,Customer says calls drop during office hours.,...,6,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0



Final Report Shape: (4901, 66)

CSV report exported: jio_complaint_priority_report.csv
Excel operations summary exported: jio_careops_operations_summary.xlsx

========== EXPORTED FILES ==========


,File Name,File Size (KB),Export Status
0,jio_complaint_priority_report.csv,6002.22,Successful
1,jio_careops_operations_summary.xlsx,1695.44,Successful



MILESTONE 10 VALIDATION COMPLETED SUCCESSFULLY!
Total Complaints: 4901
CSV Export: jio_complaint_priority_report.csv
Excel Export: jio_careops_operations_summary.xlsx
Total Business Findings: 5
All final validation checks passed!


In [12]:
# All P0 cases
filter_cases(priority="P0")

# P0 and P1 cases
filter_cases(priority=["P0", "P1"])

# All complaints from West Bengal
filter_cases(circle="West Bengal")

# Complaints from Kolkata
filter_cases(city="Kolkata")

# Network-related complaints
filter_cases(issue_group="Network")

# P0 complaints from Kolkata
filter_cases(
    priority="P0",
    city="Kolkata"
)

# Search by complaint ID
filter_cases(
    complaint_ids=["CMP000001", "CMP000009"]
)

,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,...,customer_complaint_count,severe_network_signal,recharge_risk_signal,service_followup_signal,repeat_contact_signal,priority_score,case_priority,recommended_action,ai_prompt_preview,priority_rank
0,CMP000009,CUST01060,SITE0443,2026-05-09,12:36,Ahmedabad,Gujarat,WhatsApp Bot,Unresolved Complaint,Payment was debited but recharge benefit is no...,...,4,True,True,True,True,4,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0
1,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,Unknown,SIM activation or verification is pending afte...,...,4,True,True,False,True,3,P0,Escalate to Network Operations. Investigate th...,You are a telecom customer operations analyst....,0


### Handover findings · evidence-based observations

The following observations are calculated from the supplied synthetic training data and the completed notebook outputs.

1. **Complaint volume:** The cleaned complaint table contains **4,901 unique complaints** after removing duplicate complaint IDs.
2. **Priority workload:** **3,360 complaints (68.56%)** are classified as P0 or P1 under the project’s four-signal scoring rules. This is a triage workload estimate, not a statement that all these cases are confirmed emergencies.
3. **Circle concentration:** **Gujarat** has the highest complaint volume, with **779 complaints**. Circle managers could begin their workload review here, while still checking severity and local context.
4. **Most common risk signal:** **Repeat Contact** is present in **4,438 complaints**. This high count suggests the repeat-contact rule is a major driver of prioritization and should be reviewed with business owners before operational deployment.
5. **City and issue concentration:** **Indore** is the highest-volume city with **452 complaints**, and **Network** is the largest issue group with **1,719 complaints**. A useful next step is to review site-level incident patterns for the affected cities and circles.

### Data-quality limitations and operational recommendations

- The complaint source contains missing/unknown issue categories and statuses; the cleaning stage records these as `Unknown` rather than guessing the correct label.
- Recharge data contains **642 records with unknown recharge status** and **389 missing recharge amounts**. These should be reconciled against an authoritative transaction source before refund decisions are made.
- The service-request source contains **3 orphan complaint references**, and the site inventory has **29 unmatched site references**. These records need source-system reconciliation.
- The 24-hour SLA review threshold is an illustrative project assumption and should be replaced with an approved service policy before real use.
- Risk signals overlap, so their counts must **not** be added together to estimate unique complaints. Priority should be reviewed with the underlying case evidence by authorized operations staff.

**Recommended operational next step:** Use the P0/P1 queue as a review starting point, validate the underlying network/recharge/service evidence, and prioritize data reconciliation for unknown recharge statuses and unmatched site references before treating the report as an operational source of truth.


### Final milestone · Automated assessment
**Important:** Colab retains previously created variables and files. After changing or deleting any milestone code, choose **Runtime → Restart session**, then **Run all** before trusting the self-check. Running only the assessment cell can show incorrect passes. For official grading, the instructor must execute your saved `.ipynb` in a fresh kernel with clean input files.

Run the self-check **after all ten milestones**. It creates a scorecard for your notebook using observable DataFrame outputs and files. It does **not** require an answer key to be shared with students.

**How to read results:** `Pass` means the available output meets the named structural or business-rule check; `Review` means something is absent or differs from the expected contract. This is **formative self-checking**, not a secure examination: notebook code can be modified and passing checks do not certify every analysis table or commentary. For official grading, retain a human review of findings, data quality decisions, and a sample of calculations.

**Student instruction:** Run all cells from the top before evaluating. Do not edit the assessment cell.

In [13]:

# AUTOMATED MILESTONE ASSESSMENT — Run only after finishing Milestones 1–10.
# This is a non-secure learning aid; see the project brief for review expectations.

import pandas as pd
from pathlib import Path
from IPython.display import display

_scope = globals()
_checks = []

def check(task, criterion, fn):
    try:
        passed = bool(fn())
        msg = 'Meets check' if passed else 'Missing or inconsistent output'
    except Exception as ex:
        passed = False
        msg = f'Cannot verify: {type(ex).__name__}'
    _checks.append({
        'Milestone': task,
        'Check': criterion,
        'Result': 'Pass' if passed else 'Review',
        'Points': 5 if passed else 0,
        'Feedback': msg
    })

def df(name):
    obj = _scope.get(name)
    if not isinstance(obj, pd.DataFrame):
        raise ValueError(f'{name} is not a DataFrame')
    return obj

def has_cols(name, cols):
    return set(cols).issubset(df(name).columns)

def unique_nonnull(frame, key):
    return key in frame and frame[key].notna().all() and frame[key].is_unique

def numeric(frame, col):
    return col in frame and pd.api.types.is_numeric_dtype(frame[col])

def real_dates(frame, cols):
    return all(
        col in frame and pd.api.types.is_datetime64_any_dtype(frame[col])
        for col in cols
    )

def source_counts_match(summary_name, source_name, key, count_col):
    summary = df(summary_name)
    source = df(source_name)
    expected = source[key].notna().sum()
    return (
        unique_nonnull(summary, key)
        and numeric(summary, count_col)
        and summary[count_col].sum() == expected
    )

def network_counts_match():
    n = df('network_clean')
    s = df('network_by_site')
    if not has_cols('network_by_site', [
        'cell_site_id', 'network_report_count',
        'severe_network_report_count'
    ]):
        return False
    expected = (
        n.dropna(subset=['cell_site_id'])
        .groupby('cell_site_id').size()
    )
    actual = s.set_index('cell_site_id')['network_report_count']
    return (
        unique_nonnull(s, 'cell_site_id')
        and actual.reindex(expected.index).eq(expected).all()
        and actual.sum() == len(n.dropna(subset=['cell_site_id']))
    )

def issues_match():
    cases = df('final_cases_df')
    for source_name, count_col in [
        ('recharge_clean', 'recharge_count'),
        ('requests_clean', 'request_count')
    ]:
        source = df(source_name)
        if count_col not in cases.columns:
            return False
        counts = (
            source[source.complaint_id.isin(cases.complaint_id)]
            .groupby('complaint_id').size()
        )
        actual = cases.set_index('complaint_id')[count_col].fillna(0)
        expected = counts.reindex(actual.index, fill_value=0)
        if not actual.eq(expected).all():
            return False
    return True

def flags_match():
    c = df('final_cases_df')
    flags = [
        'severe_network_signal',
        'recharge_risk_signal',
        'service_followup_signal',
        'repeat_contact_signal'
    ]
    if not has_cols('final_cases_df', flags):
        return False
    if any(not pd.api.types.is_bool_dtype(c[x]) for x in flags):
        return False

    network_expected = (
        c.severe_network_report_count.fillna(0).gt(0)
        | c.site_status.astype('string').str.strip()
          .str.casefold().isin(['down', 'inactive', 'outage'])
    ).fillna(False)

    recharge_expected = c.recharge_risk_count.fillna(0).gt(0)

    service_expected = (
        c.escalated_request_count.fillna(0).gt(0)
        | c.sla_review_count.fillna(0).gt(0)
    )

    repeat_expected = c.customer_complaint_count.fillna(0).ge(3)

    return all(
        c[col].fillna(False).eq(expected).all()
        for col, expected in zip(
            flags,
            [
                network_expected,
                recharge_expected,
                service_expected,
                repeat_expected
            ]
        )
    )

def priorities_match():
    c = df('final_cases_df')
    flags = [
        'severe_network_signal',
        'recharge_risk_signal',
        'service_followup_signal',
        'repeat_contact_signal'
    ]
    if not has_cols('final_cases_df', flags + [
        'priority_score', 'case_priority'
    ]):
        return False
    if any(not pd.api.types.is_bool_dtype(c[x]) for x in flags):
        return False

    signals = c[flags].sum(axis=1)
    expected = signals.map(
        lambda x: 'P0' if x >= 3 else (
            'P1' if x == 2 else ('P2' if x == 1 else 'P3')
        )
    )
    actual = (
        c.case_priority.astype('string').str.strip()
        .str.extract(r'^(P[0-3])', expand=False)
    )
    return (
        c.priority_score.eq(signals).all()
        and actual.eq(expected).all()
    )

def report_matches():
    r = df('final_report')
    c = df('final_cases_df')
    return (
        has_cols('final_report', ['complaint_id', 'case_priority'])
        and len(r) == len(c)
        and unique_nonnull(r, 'complaint_id')
        and set(r.complaint_id) == set(c.complaint_id)
        and r.set_index('complaint_id').case_priority.eq(
            c.set_index('complaint_id')
            .loc[r.complaint_id, 'case_priority'].to_numpy()
        ).all()
    )

def summary_matches():
    s = df('priority_summary')
    if not has_cols('priority_summary', [
        'case_priority', 'case_count'
    ]):
        return False
    expected = df('final_cases_df').case_priority.value_counts()
    got = s.groupby('case_priority').case_count.sum()
    return (
        got.reindex(expected.index, fill_value=0).eq(expected).all()
        and got.sum() == len(df('final_cases_df'))
    )

check(1, 'Complaint source loaded, key fields present',
      lambda: has_cols('complaints_df',
      ['complaint_id', 'customer_id', 'cell_site_id', 'complaint_status']))

check(1, 'Complaint source is non-empty',
      lambda: len(df('complaints_df')) > 0)

check(2, 'Network and recharge sources loaded',
      lambda: has_cols('network_df',
      ['network_report_id', 'cell_site_id', 'severity'])
      and has_cols('recharge_df',
      ['recharge_id', 'complaint_id', 'recharge_status'])
      and len(df('network_df')) > 0
      and len(df('recharge_df')) > 0)

check(2, 'Service request source loaded',
      lambda: has_cols('requests_df',
      ['service_request_id', 'complaint_id', 'service_status'])
      and len(df('requests_df')) > 0)

check(3, 'One clean record per complaint',
      lambda: unique_nonnull(df('complaints_clean'), 'complaint_id')
      and len(df('complaints_clean')) <= len(df('complaints_df')))

check(3, 'Complaint dates and categories standardized',
      lambda: real_dates(df('complaints_clean'), ['complaint_date'])
      and has_cols('complaints_clean',
      ['issue_group', 'complaint_status', 'city', 'circle']))

check(4, 'Operational records cleaned',
      lambda: unique_nonnull(df('network_clean'), 'network_report_id')
      and unique_nonnull(df('recharge_clean'), 'recharge_id')
      and unique_nonnull(df('requests_clean'), 'service_request_id'))

check(4, 'Operational dates standardized',
      lambda: real_dates(df('network_clean'), ['report_date'])
      and real_dates(df('recharge_clean'), ['recharge_date'])
      and real_dates(df('requests_clean'), ['request_created_date']))

check(5, 'Network reports aggregated by site',
      lambda: unique_nonnull(df('network_by_site'), 'cell_site_id')
      and has_cols('network_by_site',
      ['network_report_count', 'severe_network_report_count']))

check(5, 'Network report totals reconcile',
      network_counts_match)

check(6, 'Recharge summary uses unique complaint IDs',
      lambda: unique_nonnull(df('recharge_by_complaint'), 'complaint_id')
      and has_cols('recharge_by_complaint',
      ['recharge_count', 'recharge_risk_count']))

check(6, 'Recharge totals match cleaned source',
      lambda: source_counts_match(
          'recharge_by_complaint', 'recharge_clean',
          'complaint_id', 'recharge_count'
      ))

check(7, 'Service requests summarized by complaint',
      lambda: unique_nonnull(df('requests_by_complaint'), 'complaint_id')
      and has_cols('requests_by_complaint',
      ['request_count', 'open_request_count',
       'escalated_request_count', 'sla_review_count'])
      and source_counts_match(
          'requests_by_complaint', 'requests_clean',
          'complaint_id', 'request_count'
      ))

check(7, 'Repeat customer counts reconcile',
      lambda: has_cols('customer_repeat_count',
      ['customer_id', 'customer_complaint_count'])
      and unique_nonnull(df('customer_repeat_count'), 'customer_id')
      and df('customer_repeat_count').customer_complaint_count.sum()
      == df('complaints_clean').customer_id.notna().sum())

check(8, 'Master table contains all complaints once',
      lambda: unique_nonnull(df('final_cases_df'), 'complaint_id')
      and set(df('final_cases_df').complaint_id)
      == set(df('complaints_clean').complaint_id))

check(8, 'Master issue counts reconcile',
      issues_match)

check(9, 'Operational risk signals follow business rules',
      flags_match)

check(9, 'Priority mapping matches risk signals',
      priorities_match)

check(10, 'Detailed report and export exist',
      lambda: report_matches()
      and (Path(DATA_DIR) / 'jio_complaint_priority_report.csv').is_file())

check(10, 'Priority summary and export exist',
      lambda: summary_matches()
      and (Path(DATA_DIR) / 'jio_careops_operations_summary.xlsx').is_file())

assessment_df = pd.DataFrame(_checks)

score_by_task = assessment_df.groupby(
    'Milestone', as_index=False
).agg(
    Points=('Points', 'sum'),
    Passed=('Result', lambda x: (x == 'Pass').sum())
)

score_by_task['Maximum'] = 10
score_by_task['Status'] = score_by_task['Passed'].map({
    0: 'Review',
    1: 'Partially verified',
    2: 'Checks passed'
})

print('AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE')
display(score_by_task[['Milestone', 'Points', 'Maximum', 'Status']])

print(f'Automated checks score: {assessment_df.Points.sum()}/100')

print('Checks needing attention:')
display(
    assessment_df.loc[
        assessment_df.Result == 'Review',
        ['Milestone', 'Check', 'Feedback']
    ].reset_index(drop=True)
)

# Machine-readable output for independent, fresh-kernel instructor grading.
import json as _grade_json

print(
    'JIO_CAREOPS_ASSESSMENT_JSON='
    + _grade_json.dumps({
        'points': int(assessment_df.Points.sum()),
        'milestones': score_by_task[
            ['Milestone', 'Points', 'Maximum', 'Status']
        ].to_dict('records'),
        'checks': assessment_df[
            ['Milestone', 'Check', 'Result', 'Points']
        ].to_dict('records')
    })
)


AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE


,Milestone,Points,Maximum,Status
0,1,10,10,Checks passed
1,2,10,10,Checks passed
2,3,10,10,Checks passed
3,4,10,10,Checks passed
4,5,10,10,Checks passed
5,6,10,10,Checks passed
6,7,10,10,Checks passed
7,8,10,10,Checks passed
8,9,10,10,Checks passed
9,10,10,10,Checks passed


Automated checks score: 100/100
Checks needing attention:


,Milestone,Check,Feedback


JIO_CAREOPS_ASSESSMENT_JSON={"points": 100, "milestones": [{"Milestone": 1, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 2, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 3, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 4, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 5, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 6, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 7, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 8, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 9, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 10, "Points": 10, "Maximum": 10, "Status": "Checks passed"}], "checks": [{"Milestone": 1, "Check": "Complaint source loaded, key fields present", "Result": "Pass", "Points": 5}, {"Milestone": 1, "Check": "Complaint source is non-empty", "Result": "Pass", "Point

### Submission handover
1. Save your completed Colab copy in Drive and share its link with the required permissions, or submit the `.ipynb` file if instructed by your instructor.
2. Include the final automated scorecard **and** your written findings in the notebook.
3. Do not submit the original three input datasets unless requested; output CSV and Excel files are generated during your run.

**Project GitHub repository:** https://github.com/Darpan2000-web/Jio-CareOps-Network-Issue-Customer-Support-Analytics